# ScrubRef v8 — ImageWoof single-seed publication notebook

A checkpoint-safe, single-seed machine-unlearning evaluation for **ImageWoof**. The notebook compares ScrubRef-Head and ScrubRef-Rep with full retraining, output suppression, retained-data fine-tuning, UNSIR, Bad Teacher, SCRUB, and SSD.

This is a clean dataset-specific run. All checkpoints, intermediate models, and result tables are saved persistently in Google Drive.


## Scientific scope

- Seed: 42
- Empirical privacy evaluation only; no formal privacy guarantee is claimed.
- All methods use the same dataset split and evaluation protocol.
- Utility, calibration, membership inference, representation similarity, linear probes, kNN, JS divergence, and prediction disagreement are exported.
- Runtime is reported only when the saved runtime file marks it valid for direct comparison.


## Execution map

1. Stage 0: Colab environment and persistent Drive setup.
2. Stage 1: configuration and function definitions.
3. Stage 2: checkpoint and dataset preflight.
4. Stage 3: prepare seed 42 and resumable class proxies.
5. Stage 4: run each method; completed methods are skipped.
6. Stage 5: extract representation features.
7. Stage 6: compute representation metrics.
8. Stage 7: build final single-seed tables and figures.
9. Stage 8: download the professor handoff package.


## Stage 0: Environment Setup & Repository Synchronization



Mounting Google Drive to persist results and code and Cloning or pulling the latest version of the `scrubref-unlearning` repository.

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

# ── 1. Ensure running in Google Colab ────────────────────────────────────────
try:
    from google.colab import drive
except ImportError as exc:
    raise RuntimeError(
        "This notebook is configured for execution in Google Colab."
    ) from exc

# ── 2. Mount Google Drive ────────────────────────────────────────────────────
DRIVE_MOUNT_POINT = Path("/content/drive")
DRIVE_ROOT = DRIVE_MOUNT_POINT / "MyDrive"

drive.mount(str(DRIVE_MOUNT_POINT))

if not DRIVE_ROOT.is_dir():
    raise RuntimeError(
        f"Google Drive failed to mount or 'MyDrive' was not found: {DRIVE_ROOT}"
    )

# ── 3. Clone or update the repository ────────────────────────────────────────
GITHUB_REPO = "https://github.com/imnima2005/scrubref-unlearning.git"
REPO_DIR = DRIVE_ROOT / "scrubref-unlearning"

def run_git(*args: str, cwd: Path | None = None) -> None:
    """Run a Git command and raise a clear error if it fails."""
    result = subprocess.run(
        ["git", *args],
        cwd=cwd,
        capture_output=True,
        text=True,
    )

    if result.returncode != 0:
        error = result.stderr.strip() or result.stdout.strip()
        raise RuntimeError(f"Git command failed: git {' '.join(args)}\n{error}")

    if result.stdout.strip():
        print(result.stdout.strip())

if not REPO_DIR.exists():
    print(f"Cloning repository into: {REPO_DIR}")
    run_git("clone", GITHUB_REPO, str(REPO_DIR))
elif (REPO_DIR / ".git").is_dir():
    print(f"Repository already exists; updating: {REPO_DIR}")
    run_git("pull", "--ff-only", cwd=REPO_DIR)
else:
    raise RuntimeError(
        f"Destination directory exists but is not a Git repository:\n{REPO_DIR}\n"
        "Please change the directory name or check the existing path."
    )

# ── 4. Set working directory and configure Python path ───────────────────────
os.chdir(REPO_DIR)

repo_path = str(REPO_DIR.resolve())
if repo_path not in sys.path:
    sys.path.insert(0, repo_path)

print("Project root:", Path.cwd())


### Environment & Dependency Check
Inspect the Colab runtime Python version, operating system, and installed core packages (NumPy, Pandas, TensorFlow, scikit-learn) to ensure full reproducibility and compatibility with the benchmark specifications.


In [ ]:
import importlib.metadata as metadata
import platform

print("Python:", sys.version.split()[0])
print("Platform:", platform.platform())
for package in ("numpy", "pandas", "scipy", "scikit-learn", "matplotlib", "tensorflow", "joblib"):
    try:
        print(f"{package}: {metadata.version(package)}")
    except metadata.PackageNotFoundError:
        print(f"{package}: not installed")


Configure runtime environment variables before importing TensorFlow.

In [ ]:
os.environ.setdefault("TF_GPU_ALLOCATOR", "cuda_malloc_async")
os.environ.setdefault("TF_FORCE_GPU_ALLOW_GROWTH", "true")

Import the standard-library, third-party used directly in this notebook.

In [ ]:
from __future__ import annotations
# Standard library
import gc
import hashlib
import json
import math
import random
import re
import shutil
import time
import zipfile
from dataclasses import asdict, dataclass, field, replace
from datetime import datetime, timezone
from typing import Dict, Iterable, List, Optional, Sequence, Tuple, Union


In [ ]:
# Third-party packages
try:
    import joblib
    import matplotlib.pyplot as plt
    import numpy as np
    import pandas as pd
    import tensorflow as tf
except ValueError as exc:
    if "numpy.dtype size changed" in str(exc):
        raise RuntimeError(
            "The Colab VM contains an incompatible NumPy/Pandas binary stack. "
            "Use Runtime > Disconnect and delete runtime, reconnect, and run this notebook "
            "without executing any package-installation cell."
        ) from exc
    raise


In [ ]:
# Third-party packages
from IPython.display import display
from sklearn import set_config as sklearn_set_config
from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    balanced_accuracy_score,
    f1_score,
    log_loss,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

try:
    for _gpu in tf.config.list_physical_devices("GPU"):
        tf.config.experimental.set_memory_growth(_gpu, True)
except RuntimeError:
    pass

sklearn_set_config(working_memory=256)

Display the main runtime details for reproducibility.

In [ ]:
print("TensorFlow:", tf.__version__)
print("GPU devices:", tf.config.list_physical_devices("GPU"))

## Stage 1: fast-publication Configuration

This stage loads the experiment configuration, selects the dataset and run mode, and prepares the runtime state used by the notebook. Configuration values are read from the project configuration files, while selected notebook-specific constants are defined separately to support the existing experiment workflow.


### Configuration

Load the experiment configuration and the project data-loading modules. The configuration file specifies the run mode and dataset settings; the selected values are then made available to the notebook modules that require them.


In [ ]:
from src.config import ExperimentConfig
from src import config as config_module
from src import data_loader as data_loader_module

CFG = ExperimentConfig.from_yaml(REPO_DIR / "configs" / "fast_publication" / "imagewoof.yaml")
CFG.run_mode = "fast_publication"
CFG.apply_mode()

# Make CFG available inside the source modules
config_module.CFG = CFG
data_loader_module.CFG = CFG

# Make other notebook-level variables available if needed
data_loader_module.IN_COLAB = True
data_loader_module.REPO_DIR = REPO_DIR

print("CFG injected successfully")
print("run_mode:", CFG.run_mode)
print("dataset:", CFG.dataset)

Define the random seeds used for repeated runs and set notebook-level constants required by the current experiment workflow. These constants are kept separate from `ExperimentConfig` because some legacy functions still depend on notebook-level values.


In [ ]:
FINAL_SEEDS: Tuple[int, ...] = (42,)
FINAL_SEED = 42

# v7 behavior constants are intentionally outside ExperimentConfig so the legacy
# fingerprint and existing Drive directory remain unchanged.
V7_AUDIT_VERSION = 7
V7_RETRAIN_EXTRA_EPOCHS = 10
V7_RETRAIN_PHASE2_LR = 2e-4

def v7_representation_methods(cfg: ExperimentConfig) -> Tuple[str, ...]:
    return tuple(dict.fromkeys((*cfg.representation_methods, "retain_finetune")))

Import the project utilities used to manage paths and artifacts, load models, and control runtime memory.

In [ ]:
from src.utils import (
    set_global_determinism,
    set_stage_status,
    stage_status_path,
    seed_dir,
    experiment_dir,
    config_fingerprint,
    atomic_write_dataframe,
    atomic_write_json,
    utc_now,
    load_saved_model,
    save_json,
    array_memory_mb,
    reset_tensorflow,
    release_memory
)

Set the global random seed to ensure reproducibility for the experiment run.


In [ ]:
set_global_determinism(CFG.seeds[0])

Initialize the state trackers (`ACTIVE_METHOD_NAME`, `ACTIVE_METHOD_DIR`, `ACTIVE_FIT_COUNTER`) used to manage the execution lifecycle and artifact organization for each unlearning method being tested.

In [ ]:
ACTIVE_METHOD_DIR: Optional[Path] = None
ACTIVE_FIT_COUNTER: int = 0
ACTIVE_METHOD_NAME: Optional[str] = None

### Dataset and RAM-safe input pipeline

In [ ]:
ArrayLike = Union[np.ndarray, Sequence[str]]

In [ ]:
from src.data_loader import (
    DataBundle,
    UnlearningSplit,
    ensure_imagewoof,
    list_directory_split,
    load_directory_bundle,
    load_dataset,
    make_unlearning_split,
    decode_item,
    augment_image,
    materialize_items
)

Some unlearning methods and datasets can exceed the available GPU memory when using the default batch size. The variables below enable method-aware, dataset-specific batch-size caps during runtime while preserving the experiment configuration.

In [ ]:
_RUNTIME_BATCH_MESSAGES: set = set()

In [ ]:
def effective_runtime_batch_size(
    bundle: DataBundle,
    requested_batch_size: int,
    training: bool,
) -> int:
    """Return a conservative batch size without changing the experiment fingerprint."""
    requested = max(1, int(requested_batch_size))
    method = globals().get("ACTIVE_METHOD_NAME")
    base_name = str(bundle.name).replace("_materialized", "")

    if base_name == "cifar100":
        if training:
            cap = 4 if method in {"unsir", "bad_teacher", "scrub"} else 8
        else:
            cap = 8
    elif base_name in {"imagewoof", "imagenet100"}:
        if training:
            cap = 2 if method in {"unsir", "bad_teacher", "scrub"} else 4
        else:
            cap = 4
    elif base_name == "cifar10":
        if training:
            cap = 16 if method in {"unsir", "bad_teacher", "scrub"} else 32
        else:
            cap = 64
    else:
        cap = 4 if training else 8

    effective = min(requested, cap)
    message_key = (base_name, method, requested, effective, training)
    if effective != requested and message_key not in _RUNTIME_BATCH_MESSAGES:
        phase = "training" if training else "evaluation"
        method_text = f", method={method}" if method else ""
        print(
            f"Runtime GPU safety cap: {base_name} {phase} batch "
            f"{requested} -> {effective}{method_text}"
        )
        _RUNTIME_BATCH_MESSAGES.add(message_key)
    return effective

In [ ]:
def make_dataset(
    items: np.ndarray,
    labels: np.ndarray,
    bundle: DataBundle,
    batch_size: int,
    shuffle: bool,
    augment: bool,
    repeat: bool = False,
    seed: int = 42,
) -> tf.data.Dataset:
    if len(items) != len(labels):
        raise ValueError(f"items/labels length mismatch: {len(items)} != {len(labels)}")

    training = bool(shuffle or augment or repeat)
    effective_batch = effective_runtime_batch_size(
        bundle,
        batch_size,
        training=training,
    )

    labels_array = np.asarray(labels, dtype=np.int64)
    ds = tf.data.Dataset.from_tensor_slices((items, labels_array))
    if shuffle:
        ds = ds.shuffle(
            min(len(items), CFG.shuffle_buffer),
            seed=seed,
            reshuffle_each_iteration=True,
        )
    if repeat:
        ds = ds.repeat()

    def mapper(item: tf.Tensor, label: tf.Tensor) -> Tuple[tf.Tensor, tf.Tensor]:
        image = decode_item(item, bundle.image_size, bundle.path_mode)
        if augment:
            image = augment_image(image, bundle.image_size)
        return image, tf.cast(label, tf.int64)

    options = tf.data.Options()
    options.experimental_deterministic = not shuffle
    try:
        options.autotune.enabled = False
        options.threading.private_threadpool_size = min(
            int(CFG.data_private_threadpool_size),
            2,
        )
        options.threading.max_intra_op_parallelism = 1
    except Exception:
        pass

    ds = ds.with_options(options)
    ds = ds.map(
        mapper,
        num_parallel_calls=min(int(CFG.data_parallel_calls), 2),
    )
    ds = ds.batch(effective_batch, drop_remainder=False)
    return ds.prefetch(min(int(CFG.data_prefetch_batches), 1))


### Model and training utilities

This section imports the model-building and training helpers used in the notebook. The model utilities provide ResNet construction, classifier-head management, weight cloning, and output conversion. The training utilities compile the model and support inference when model outputs are logits.


In [ ]:
from src.models import (
    build_resnet18,
    classifier_layer,
    clone_architecture,
    clone_with_weights,
    fresh_clone,
    leaf_layers,
    model_containers,
    build_feature_model,
    output_to_logits,
    output_to_probs,
    set_all_trainable,
    set_tail_trainable,
    custom_objects
)

In [ ]:
from src.training import (
    compile_supervised,
    infer_output_is_logits
)

In [ ]:
def train_supervised(
    model: tf.keras.Model,
    items: np.ndarray,
    labels: np.ndarray,
    bundle: DataBundle,
    output_is_logits: bool,
    epochs: int,
    batch_size: int,
    learning_rate: float,
    seed: int,
    validation: Optional[Tuple[np.ndarray, np.ndarray]] = None,
    train_all: bool = True,
    tail_layers: int = 0,
) -> tf.keras.Model:
    if train_all:
        set_all_trainable(model, True, freeze_batch_norm=False)
    else:
        set_tail_trainable(model, bundle.num_classes, tail_layers)

    compile_supervised(model, output_is_logits, learning_rate)
    train_ds = make_dataset(items, labels, bundle, batch_size, True, True, False, seed)
    validation_ds = None
    callbacks: List[tf.keras.callbacks.Callback] = []

    if validation is not None and len(validation[0]) > 0:
        validation_ds = make_dataset(validation[0], validation[1], bundle, batch_size, False, False, False, seed)
        callbacks.extend([
            tf.keras.callbacks.ReduceLROnPlateau(
                monitor="val_loss",
                factor=0.5,
                patience=max(2, CFG.early_stopping_patience // 2),
                min_lr=1e-6,
                verbose=1,
            ),
            tf.keras.callbacks.EarlyStopping(
                monitor="val_loss",
                patience=CFG.early_stopping_patience,
                restore_best_weights=True,
                verbose=1,
            ),
        ])

    if ACTIVE_METHOD_DIR is not None:
        global ACTIVE_FIT_COUNTER
        ACTIVE_FIT_COUNTER += 1
        fit_name = f"fit_{ACTIVE_FIT_COUNTER:02d}"
        backup_dir = ACTIVE_METHOD_DIR / f"keras_backup_{fit_name}"
        backup_dir.mkdir(parents=True, exist_ok=True)
        callbacks.insert(0, tf.keras.callbacks.BackupAndRestore(
            backup_dir=str(backup_dir),
            save_freq="epoch",
            delete_checkpoint=False,
        ))
        callbacks.append(tf.keras.callbacks.CSVLogger(
            str(ACTIVE_METHOD_DIR / f"{fit_name}_history.csv"),
            append=True,
        ))

    model.fit(
        train_ds,
        validation_data=validation_ds,
        epochs=epochs,
        verbose=2,
        callbacks=callbacks,
    )
    return model

def load_or_train_original(
    cfg: ExperimentConfig,
    bundle: DataBundle,
    seed: int,
    run_dir: Path,
) -> Tuple[tf.keras.Model, bool, str]:
    checkpoint = Path(cfg.checkpoint_paths[cfg.dataset])

    if checkpoint.exists():
        print(f"Loading pre-trained original model from: {checkpoint}")
        model = tf.keras.models.load_model(checkpoint, custom_objects=custom_objects(), compile=False, safe_mode=False)
        source = str(checkpoint)
    else:
        print(f"Checkpoint not found. Training a fresh ResNet-18 model for {cfg.original_epochs} epochs...")
        model = build_resnet18(bundle.input_shape, bundle.num_classes)
        sample_ds = make_dataset(bundle.train_items[:16], bundle.train_labels[:16], bundle, 16, False, False, False, seed)
        sample_x, _ = next(iter(sample_ds))

        output_is_logits = infer_output_is_logits(model, sample_x, True)
        model = train_supervised(
            model,
            bundle.train_items,
            bundle.train_labels,
            bundle,
            output_is_logits,
            cfg.original_epochs,
            cfg.batch_size,
            cfg.original_lr,
            seed,
            validation=None,
            train_all=True,
        )
        source = str(f"./checkpoints/{cfg.dataset}_trained_original.keras")
        model.save(source)
        print(f"Freshly trained original model saved to: {source}")

    sample_ds = make_dataset(bundle.train_items[:16], bundle.train_labels[:16], bundle, 16, False, False, False, seed)
    sample_x, _ = next(iter(sample_ds))
    output_is_logits = infer_output_is_logits(model, sample_x, cfg.output_is_logits)

    return model, output_is_logits, source

### Utility, calibration, membership inference, and parameter metrics

This section imports metric helpers and defines evaluation routines for the unlearning experiments. It measures model utility on the full test set and separately on retained and forgotten classes, evaluates prediction calibration, and estimates privacy leakage using membership-inference attacks. The imported parameter and GPU-memory utilities are used to track model changes and runtime resource usage.

In [ ]:
from src.metrics import (
    expected_calibration_error,
    brier_multiclass,
    per_class_accuracy,
    entropy_scores,
    modified_entropy_scores,
    safe_auc,
    parameter_change_statistics,
    peak_gpu_memory_mb
)

The functions below generate model probabilities and compute utility and privacy metrics. `evaluate_utility` reports classification performance and calibration-related measures, while `evaluate_membership_inference` compares forgotten training samples with forgotten test samples using confidence-, loss-, entropy-, and learned attacks.

In [ ]:
def predict_probabilities(
    model: tf.keras.Model,
    items: np.ndarray,
    labels: np.ndarray,
    bundle: DataBundle,
    output_is_logits: bool,
    batch_size: int,
    seed: int,
) -> Tuple[np.ndarray, np.ndarray]:
    ds = make_dataset(items, labels, bundle, batch_size, False, False, False, seed)
    probabilities: List[np.ndarray] = []
    labels_out: List[np.ndarray] = []
    for xb, yb in ds:
        output = model(xb, training=False)
        probabilities.append(output_to_probs(output, output_is_logits).numpy())
        labels_out.append(yb.numpy())
    return np.concatenate(probabilities), np.concatenate(labels_out)

def evaluate_utility(
    model: tf.keras.Model,
    bundle: DataBundle,
    forget_classes: Sequence[int],
    output_is_logits: bool,
    batch_size: int,
    seed: int,
) -> Tuple[Dict[str, float], np.ndarray]:
    probs, y_true = predict_probabilities(
        model, bundle.test_items, bundle.test_labels, bundle, output_is_logits, batch_size, seed
    )
    y_pred = probs.argmax(axis=1)
    forget_mask = np.isin(y_true, np.asarray(forget_classes))
    retain_mask = ~forget_mask
    top5 = np.mean([
        label in np.argpartition(row, -min(5, bundle.num_classes))[-min(5, bundle.num_classes):]
        for row, label in zip(probs, y_true)
    ])
    metrics = {
        "overall_accuracy": float(accuracy_score(y_true, y_pred)),
        "retained_accuracy": float(accuracy_score(y_true[retain_mask], y_pred[retain_mask])),
        "forgotten_accuracy": float(accuracy_score(y_true[forget_mask], y_pred[forget_mask])),
        "top5_accuracy": float(top5),
        "macro_f1": float(f1_score(y_true, y_pred, average="macro", zero_division=0)),
        "balanced_accuracy": float(balanced_accuracy_score(y_true, y_pred)),
        "nll": float(log_loss(y_true, np.clip(probs, 1e-7, 1.0), labels=np.arange(bundle.num_classes))),
        "brier": brier_multiclass(y_true, probs, bundle.num_classes),
        "ece_15": expected_calibration_error(y_true, probs, bins=15),
        "mean_global_forget_mass": float(np.mean(np.sum(probs[:, forget_classes], axis=1))),
        "mean_forget_test_true_class_confidence": float(
            np.mean(probs[np.where(forget_mask)[0], y_true[forget_mask]])
        ),
    }
    return metrics, per_class_accuracy(y_true, y_pred, bundle.num_classes)

def learned_mia(
    member_probs: np.ndarray,
    member_labels: np.ndarray,
    nonmember_probs: np.ndarray,
    nonmember_labels: np.ndarray,
    repeats: int,
    seed: int,
) -> Tuple[float, float, float]:
    def attack_features(probs: np.ndarray, labels: np.ndarray) -> np.ndarray:
        rows = np.arange(len(labels))
        true_conf = probs[rows, labels]
        entropy = entropy_scores(probs)
        mod_entropy = modified_entropy_scores(probs, labels)
        sorted_probs = np.sort(probs, axis=1)[:, -3:]
        return np.column_stack([true_conf, entropy, mod_entropy, sorted_probs])

    x = np.concatenate([
        attack_features(member_probs, member_labels),
        attack_features(nonmember_probs, nonmember_labels),
    ])
    y = np.concatenate([
        np.ones(len(member_probs), dtype=np.int64),
        np.zeros(len(nonmember_probs), dtype=np.int64),
    ])
    aucs: List[float] = []
    aps: List[float] = []
    for repeat in range(repeats):
        x_train, x_test, y_train, y_test = train_test_split(
            x, y, test_size=0.5, random_state=seed + repeat, stratify=y
        )
        attack = make_pipeline(StandardScaler(), LogisticRegression(max_iter=300, class_weight="balanced"))
        attack.fit(x_train, y_train)
        scores = attack.predict_proba(x_test)[:, 1]
        aucs.append(roc_auc_score(y_test, scores))
        aps.append(average_precision_score(y_test, scores))
    raw_auc = float(np.mean(aucs))
    corrected_auc = float(max(raw_auc, 1.0 - raw_auc))
    return raw_auc, corrected_auc, float(np.mean(aps))

def evaluate_membership_inference(
    model: tf.keras.Model,
    split: UnlearningSplit,
    bundle: DataBundle,
    output_is_logits: bool,
    batch_size: int,
    repeats: int,
    seed: int,
) -> Dict[str, float]:
    member_probs, member_labels = predict_probabilities(
        model,
        split.forget_train_items,
        split.forget_train_labels,
        bundle,
        output_is_logits,
        batch_size,
        seed,
    )
    nonmember_probs, nonmember_labels = predict_probabilities(
        model,
        split.forget_test_items,
        split.forget_test_labels,
        bundle,
        output_is_logits,
        batch_size,
        seed,
    )
    n = min(len(member_probs), len(nonmember_probs))
    rng = np.random.default_rng(seed)
    member_idx = rng.choice(len(member_probs), n, replace=False)
    nonmember_idx = rng.choice(len(nonmember_probs), n, replace=False)
    member_probs = member_probs[member_idx]
    member_labels = member_labels[member_idx]
    nonmember_probs = nonmember_probs[nonmember_idx]
    nonmember_labels = nonmember_labels[nonmember_idx]

    probs = np.concatenate([member_probs, nonmember_probs])
    labels = np.concatenate([member_labels, nonmember_labels])
    membership = np.concatenate([np.ones(n), np.zeros(n)])
    rows = np.arange(len(labels))
    true_conf = probs[rows, labels]
    negative_loss = np.log(np.clip(true_conf, 1e-7, 1.0))
    negative_entropy = -entropy_scores(probs)
    negative_modified_entropy = -modified_entropy_scores(probs, labels)
    learned_auc_raw, learned_auc, learned_ap = learned_mia(
        member_probs, member_labels, nonmember_probs, nonmember_labels, repeats, seed
    )
    return {
        "mia_confidence_auc": safe_auc(membership, true_conf),
        "mia_loss_auc": safe_auc(membership, negative_loss),
        "mia_entropy_auc": safe_auc(membership, negative_entropy),
        "mia_modified_entropy_auc": safe_auc(membership, negative_modified_entropy),
        "mia_learned_auc_raw": learned_auc_raw,
        "mia_learned_auc": learned_auc,
        "mia_learned_ap": learned_ap,
        "mia_member_count": float(n),
    }



### Fast representation audits

This section defines lightweight audits for comparing the model’s learned representations before and after unlearning. To keep the analysis practical, `stratified_indices` selects a reproducible, class-balanced subset, and `extract_features` obtains feature vectors in batches through the TensorFlow input pipeline.

The audit utilities measure representation similarity with linear CKA, assess how informative the extracted features remain using linear-probe and k-NN classifiers, and compare output distributions with Jensen–Shannon divergence.

In [ ]:
def stratified_indices(labels: np.ndarray, limit: int, seed: int) -> np.ndarray:
    if len(labels) <= limit:
        return np.arange(len(labels))
    rng = np.random.default_rng(seed)
    unique = np.unique(labels)
    per_class = max(1, limit // len(unique))
    selected: List[int] = []
    for class_id in unique:
        candidates = np.where(labels == class_id)[0]
        count = min(per_class, len(candidates))
        selected.extend(rng.choice(candidates, count, replace=False).tolist())
    if len(selected) < limit:
        remaining = np.setdiff1d(np.arange(len(labels)), np.asarray(selected), assume_unique=False)
        extra = rng.choice(remaining, min(limit - len(selected), len(remaining)), replace=False)
        selected.extend(extra.tolist())
    return np.asarray(selected[:limit], dtype=np.int64)


def extract_features(
    model: tf.keras.Model,
    items: np.ndarray,
    labels: np.ndarray,
    bundle: DataBundle,
    batch_size: int,
    seed: int,
) -> Tuple[np.ndarray, np.ndarray]:
    feature_model = build_feature_model(model, bundle.num_classes)
    ds = make_dataset(items, labels, bundle, batch_size, False, False, False, seed)
    features: List[np.ndarray] = []
    labels_out: List[np.ndarray] = []
    for xb, yb in ds:
        feat = tf.cast(feature_model(xb, training=False), tf.float32)
        feat = tf.reshape(feat, [tf.shape(feat)[0], -1])
        features.append(feat.numpy())
        labels_out.append(yb.numpy())
    features_array = np.concatenate(features).astype(np.float32, copy=False)
    labels_array = np.concatenate(labels_out).astype(np.int64, copy=False)
    del feature_model, features, labels_out
    release_memory(clear_session=False)
    return features_array, labels_array


def linear_cka(x: np.ndarray, y: np.ndarray) -> float:
    x = np.asarray(x, dtype=np.float64)
    y = np.asarray(y, dtype=np.float64)
    x = x - x.mean(axis=0, keepdims=True)
    y = y - y.mean(axis=0, keepdims=True)
    k = x @ x.T
    l = y @ y.T
    n = k.shape[0]
    h = np.eye(n) - np.ones((n, n), dtype=np.float64) / n
    kc = h @ k @ h
    lc = h @ l @ h
    numerator = float(np.sum(kc * lc))
    denominator = math.sqrt(float(np.sum(kc * kc)) * float(np.sum(lc * lc)))
    return numerator / max(denominator, 1e-12)

def representation_probe_metrics(
    train_features: np.ndarray,
    train_labels: np.ndarray,
    test_features: np.ndarray,
    test_labels: np.ndarray,
    forget_classes: Sequence[int],
) -> Dict[str, float]:
    train_features = np.asarray(train_features, dtype=np.float32)
    test_features = np.asarray(test_features, dtype=np.float32)

    linear_probe = make_pipeline(
        StandardScaler(copy=True),
        SGDClassifier(
            loss="log_loss",
            alpha=1e-4,
            max_iter=500,
            tol=1e-3,
            early_stopping=True,
            validation_fraction=0.1,
            n_iter_no_change=10,
            class_weight="balanced",
            random_state=42,
            n_jobs=-1,
        ),
    )
    linear_probe.fit(train_features, train_labels)
    linear_pred = linear_probe.predict(test_features)

    knn = make_pipeline(
        StandardScaler(copy=True),
        KNeighborsClassifier(
            n_neighbors=5,
            weights="distance",
            algorithm="brute",
            n_jobs=-1,
        ),
    )
    knn.fit(train_features, train_labels)
    knn_pred = knn.predict(test_features)

    forget_mask = np.isin(test_labels, np.asarray(forget_classes))
    retain_mask = ~forget_mask
    metrics = {
        "linear_probe_overall_accuracy": float(accuracy_score(test_labels, linear_pred)),
        "linear_probe_forgotten_accuracy": float(accuracy_score(test_labels[forget_mask], linear_pred[forget_mask])),
        "linear_probe_retained_accuracy": float(accuracy_score(test_labels[retain_mask], linear_pred[retain_mask])),
        "knn_overall_accuracy": float(accuracy_score(test_labels, knn_pred)),
        "knn_forgotten_accuracy": float(accuracy_score(test_labels[forget_mask], knn_pred[forget_mask])),
        "knn_retained_accuracy": float(accuracy_score(test_labels[retain_mask], knn_pred[retain_mask])),
    }
    del linear_probe, knn, linear_pred, knn_pred
    release_memory(clear_session=False)
    return metrics


def jensen_shannon_divergence(p: np.ndarray, q: np.ndarray) -> float:
    p = np.clip(p, 1e-7, 1.0)
    q = np.clip(q, 1e-7, 1.0)
    m = 0.5 * (p + q)
    kl_pm = np.sum(p * (np.log(p) - np.log(m)), axis=1)
    kl_qm = np.sum(q * (np.log(q) - np.log(m)), axis=1)
    return float(np.mean(0.5 * (kl_pm + kl_qm)))


### Resumable class-proxy generation

This section generates synthetic inputs for the forgotten classes without requiring access to their original training data. `generate_class_proxies` optimizes images against a frozen teacher model to produce class-conditioned proxies. It saves completed batches and the final proxy set to disk, allowing generation to resume from cached work if execution is interrupted.

`generate_unsir_error_maximizing_noise` creates synthetic noise in small batches to limit GPU memory use. It maximizes the teacher’s classification loss for balanced forgotten-class targets and returns the generated noise together with its labels. These proxies and noise samples are used by the corresponding unlearning procedures.


In [ ]:
from src.proxy_generation import balanced_proxy_targets

def generate_class_proxies(
    teacher: tf.keras.Model,
    bundle: DataBundle,
    forget_classes: Sequence[int],
    output_is_logits: bool,
    cfg: ExperimentConfig,
    seed: int,
    cache_path: Path,
) -> Tuple[np.ndarray, np.ndarray, Dict[str, float]]:
    if cache_path.exists():
        with np.load(cache_path, allow_pickle=True) as cached:
            images = cached["images"].astype(np.float32, copy=False)
            labels = cached["labels"].astype(np.int64, copy=False)
            metadata = json.loads(str(cached["metadata"].item()))
        print("Reusing completed proxies:", cache_path)
        return images, labels, metadata

    set_global_determinism(seed)
    teacher.trainable = False
    batch_dir = cache_path.parent / "proxy_batches"
    batch_dir.mkdir(parents=True, exist_ok=True)
    target_schedule = balanced_proxy_targets(forget_classes, cfg.proxy_count)
    total_batches = math.ceil(cfg.proxy_count / cfg.proxy_batch_size)

    for batch_index in range(total_batches):
        start = batch_index * cfg.proxy_batch_size
        end = min(cfg.proxy_count, start + cfg.proxy_batch_size)
        batch_path = batch_dir / f"batch_{batch_index:04d}.npz"
        if batch_path.exists():
            print(f"Proxy batch {batch_index + 1}/{total_batches}: cached")
            continue

        targets_np = target_schedule[start:end]
        targets = tf.convert_to_tensor(targets_np, dtype=tf.int64)
        images = tf.Variable(
            tf.random.uniform((len(targets_np),) + bundle.input_shape, 0.0, 255.0, seed=seed + start)
        )
        optimizer = tf.keras.optimizers.Adam(cfg.proxy_lr)
        executed_steps = 0
        mean_target_probability = 0.0

        for step_index in range(cfg.proxy_steps):
            with tf.GradientTape() as tape:
                output = teacher(images, training=False)
                logits = output_to_logits(output, output_is_logits)
                class_loss = tf.reduce_mean(
                    tf.keras.losses.sparse_categorical_crossentropy(targets, logits, from_logits=True)
                )
                normalized = images / 255.0
                tv_loss = tf.reduce_mean(tf.image.total_variation(normalized))
                l2_loss = tf.reduce_mean(tf.square(normalized - 0.5))
                loss = class_loss + cfg.proxy_tv_weight * tv_loss + cfg.proxy_l2_weight * l2_loss
            gradients = tape.gradient(loss, images)
            optimizer.apply_gradients([(gradients, images)])
            images.assign(tf.clip_by_value(images, 0.0, 255.0))
            executed_steps = step_index + 1

            should_check = (
                executed_steps >= cfg.proxy_min_steps
                and executed_steps % cfg.proxy_check_every == 0
            )
            if should_check:
                probabilities = output_to_probs(teacher(images, training=False), output_is_logits)
                selected = tf.gather_nd(
                    probabilities,
                    tf.stack([tf.range(len(targets_np), dtype=tf.int64), targets], axis=1),
                )
                mean_target_probability = float(tf.reduce_mean(selected).numpy())
                if mean_target_probability >= cfg.proxy_target_probability:
                    break

        probabilities = output_to_probs(teacher(images, training=False), output_is_logits).numpy()
        selected_probabilities = probabilities[np.arange(len(targets_np)), targets_np]
        np.savez_compressed(
            batch_path,
            images=images.numpy().astype(np.float32),
            labels=targets_np.astype(np.int64),
            target_probabilities=selected_probabilities.astype(np.float32),
            executed_steps=np.asarray([executed_steps], dtype=np.int64),
        )
        print(
            f"Proxy batch {batch_index + 1}/{total_batches}: "
            f"steps={executed_steps}, mean_target_p={selected_probabilities.mean():.4f}"
        )
        del images, optimizer, probabilities, selected_probabilities
        release_memory(clear_session=False)

    all_images: List[np.ndarray] = []
    all_labels: List[np.ndarray] = []
    all_probabilities: List[np.ndarray] = []
    all_steps: List[int] = []
    for batch_index in range(total_batches):
        batch_path = batch_dir / f"batch_{batch_index:04d}.npz"
        if not batch_path.exists():
            raise RuntimeError(f"Missing proxy batch checkpoint: {batch_path}")
        with np.load(batch_path) as cached:
            all_images.append(cached["images"].astype(np.float32, copy=False))
            all_labels.append(cached["labels"].astype(np.int64, copy=False))
            all_probabilities.append(cached["target_probabilities"].astype(np.float32, copy=False))
            all_steps.append(int(cached["executed_steps"][0]))

    proxy_images = np.concatenate(all_images, axis=0)[:cfg.proxy_count]
    proxy_labels = np.concatenate(all_labels, axis=0)[:cfg.proxy_count]
    target_probabilities = np.concatenate(all_probabilities, axis=0)[:cfg.proxy_count]
    metadata = {
        "mean_target_probability": float(np.mean(target_probabilities)),
        "min_target_probability": float(np.min(target_probabilities)),
        "max_target_probability": float(np.max(target_probabilities)),
        "count": float(len(proxy_images)),
        "mean_optimization_steps": float(np.mean(all_steps)),
        "total_batches": float(total_batches),
    }
    cache_path.parent.mkdir(parents=True, exist_ok=True)
    np.savez_compressed(
        cache_path,
        images=proxy_images,
        labels=proxy_labels,
        metadata=json.dumps(metadata),
    )
    print("Completed proxy set:", metadata)
    return proxy_images, proxy_labels, metadata

def generate_unsir_error_maximizing_noise(
    teacher: tf.keras.Model,
    bundle: DataBundle,
    forget_classes: Sequence[int],
    output_is_logits: bool,
    count: int,
    steps: int,
    seed: int,
) -> Tuple[np.ndarray, np.ndarray]:
    """Generate UNSIR noise in tiny batches to avoid EfficientNet activation OOM."""
    set_global_determinism(seed)
    teacher.trainable = False
    targets_np = balanced_proxy_targets(forget_classes, count)
    base_name = str(bundle.name).replace("_materialized", "")
    if base_name == "cifar100":
        micro_batch = 4
    elif base_name in {"imagewoof", "imagenet100"}:
        micro_batch = 2
    else:
        micro_batch = 16

    noise_parts: List[np.ndarray] = []
    label_parts: List[np.ndarray] = []
    total_batches = math.ceil(count / micro_batch)

    for batch_index, start in enumerate(range(0, count, micro_batch), start=1):
        end = min(start + micro_batch, count)
        local_targets_np = targets_np[start:end]
        local_targets = tf.convert_to_tensor(local_targets_np, dtype=tf.int64)
        stateless_seed = tf.constant([seed, batch_index], dtype=tf.int32)
        initial = tf.random.stateless_normal(
            (end - start,) + bundle.input_shape,
            seed=stateless_seed,
            mean=127.5,
            stddev=40.0,
            dtype=tf.float32,
        )
        noise = tf.Variable(initial)
        optimizer = tf.keras.optimizers.Adam(1.0)

        for _ in range(steps):
            with tf.GradientTape() as tape:
                output = teacher(noise, training=False)
                logits = output_to_logits(output, output_is_logits)
                ce = tf.reduce_mean(
                    tf.keras.losses.sparse_categorical_crossentropy(
                        local_targets,
                        logits,
                        from_logits=True,
                    )
                )
                regularizer = 1e-6 * tf.reduce_mean(tf.square(noise - 127.5))
                objective = -ce + regularizer
            gradient = tape.gradient(objective, noise)
            optimizer.apply_gradients([(gradient, noise)])
            noise.assign(tf.clip_by_value(noise, 0.0, 255.0))

        noise_parts.append(noise.numpy().astype(np.float32, copy=False))
        label_parts.append(local_targets_np.astype(np.int64, copy=False))
        print(
            f"UNSIR noise batch {batch_index}/{total_batches} complete "
            f"(size={end - start}, steps={steps})"
        )
        del optimizer, noise, initial, local_targets
        gc.collect()

    return np.concatenate(noise_parts, axis=0), np.concatenate(label_parts, axis=0)




### Corrected ScrubRef with epoch checkpoints


This section defines the objective functions and distribution-shaping operations used during unlearning:
- **`masked_distribution`**: Computes temperature-scaled probabilities over retained classes only by masking out forgotten class logits.
- **`mixup_batch`**: Implements Mixup data augmentation to regularize retained data training.
- **`forgot_mass` & `retained_entropy`**: Quantify the unlearning objectives by measuring total probability mass assigned to forgotten classes and dispersion across remaining retained classes.
- **`apply_logit_surgery`**: Suppresses the classifier weights and applies a negative bias shift to the forgotten class heads before retained refinement.


In [ ]:
def masked_distribution(
    output: tf.Tensor,
    output_is_logits: bool,
    forget_classes: Sequence[int],
    temperature: float,
) -> tf.Tensor:
    probs = output_to_probs(output, output_is_logits, temperature)
    mask = tf.ones(tf.shape(probs)[-1], dtype=tf.float32)
    indices = tf.reshape(tf.convert_to_tensor(forget_classes, dtype=tf.int32), [-1, 1])
    mask = tf.tensor_scatter_nd_update(mask, indices, tf.zeros(len(forget_classes), tf.float32))
    probs = probs * mask
    return probs / tf.maximum(tf.reduce_sum(probs, axis=-1, keepdims=True), 1e-7)


def mixup_batch(images: tf.Tensor, labels: tf.Tensor, num_classes: int, alpha: float) -> Tuple[tf.Tensor, tf.Tensor]:
    labels_one_hot = tf.one_hot(labels, num_classes, dtype=tf.float32)
    if alpha <= 0.0:
        return images, labels_one_hot
    gamma_a = tf.random.gamma([], alpha)
    gamma_b = tf.random.gamma([], alpha)
    lam = gamma_a / (gamma_a + gamma_b)
    permutation = tf.random.shuffle(tf.range(tf.shape(images)[0]))
    mixed_images = lam * images + (1.0 - lam) * tf.gather(images, permutation)
    mixed_labels = lam * labels_one_hot + (1.0 - lam) * tf.gather(labels_one_hot, permutation)
    return mixed_images, mixed_labels

def categorical_ce_from_output(targets: tf.Tensor, output: tf.Tensor, output_is_logits: bool) -> tf.Tensor:
    if output_is_logits:
        return tf.reduce_mean(tf.keras.losses.categorical_crossentropy(targets, output, from_logits=True))
    return tf.reduce_mean(tf.keras.losses.categorical_crossentropy(targets, tf.clip_by_value(output, 1e-7, 1.0), from_logits=False))

def forgot_mass(probs: tf.Tensor, forget_classes: Sequence[int]) -> tf.Tensor:
    selected = tf.gather(probs, tf.convert_to_tensor(forget_classes, dtype=tf.int32), axis=1)
    return tf.reduce_sum(selected, axis=1)

def retained_entropy(probs: tf.Tensor, forget_classes: Sequence[int]) -> tf.Tensor:
    mask = tf.ones(tf.shape(probs)[-1], dtype=tf.float32)
    indices = tf.reshape(tf.convert_to_tensor(forget_classes, dtype=tf.int32), [-1, 1])
    mask = tf.tensor_scatter_nd_update(mask, indices, tf.zeros(len(forget_classes), tf.float32))
    retained = probs * mask
    retained = retained / tf.maximum(tf.reduce_sum(retained, axis=1, keepdims=True), 1e-7)
    retained = tf.clip_by_value(retained, 1e-7, 1.0)
    return -tf.reduce_sum(retained * tf.math.log(retained), axis=1)

def gradient_global_norm(gradients: Sequence[Optional[tf.Tensor]]) -> tf.Tensor:
    valid = [tf.cast(g, tf.float32) for g in gradients if g is not None]
    return tf.linalg.global_norm(valid) if valid else tf.constant(0.0, tf.float32)

def apply_logit_surgery(
    model: tf.keras.Model,
    num_classes: int,
    forget_classes: Sequence[int],
    shift: float,
) -> None:
    classifier = classifier_layer(model, num_classes)
    weights = classifier.get_weights()
    if not weights:
        raise ValueError("Classifier layer has no weights.")
    kernel = weights[0].copy()
    kernel[:, list(forget_classes)] *= 0.25
    if len(weights) == 2:
        bias = weights[1].copy()
        bias[list(forget_classes)] -= shift
        classifier.set_weights([kernel, bias])
    else:
        classifier.set_weights([kernel])


This section executes the main unlearning pipeline across two distinct phases:
1. **Representation Scrubbing (`scrub_epoch`)**: Jointly updates the student network using teacher-guided retained distillation and proxy-driven forgetting (output mass suppression and optional representation-level alignment via `forget_feature_loss`).
2. **Retained Refinement (`refine_epoch`)**: Fine-tunes the network on retained data via supervised cross-entropy and masked KD to recover utility after optional logit surgery.
3. **Confidence Clamping (`confidence_clamp`)**: Post-hoc optimization pass that enforces strict probability thresholds on forgotten classes using proxies.

State variables (`scrub_epoch_var`, `refine_epoch_var`) and `tf.train.CheckpointManager` allow the entire training process to resume seamlessly from disk if interrupted.


In [ ]:
from src.scrubref import categorical_kl

def scrubref_train(
    original_model: tf.keras.Model,
    split: UnlearningSplit,
    bundle: DataBundle,
    proxies: np.ndarray,
    proxy_labels: np.ndarray,
    forget_classes: Sequence[int],
    output_is_logits: bool,
    cfg: ExperimentConfig,
    seed: int,
    representation_aware: bool,
) -> Tuple[tf.keras.Model, Dict[str, float]]:
    set_global_determinism(seed)
    student = clone_with_weights(original_model)
    teacher = clone_with_weights(original_model)
    teacher.trainable = False
    tail_layers = cfg.representation_tail_layers if representation_aware else cfg.head_tail_layers
    trainable_names = set_tail_trainable(student, bundle.num_classes, tail_layers)

    optimizer = tf.keras.optimizers.Adam(cfg.scrub_lr)
    optimizer_refine = tf.keras.optimizers.Adam(cfg.refine_lr)
    scrub_epoch_var = tf.Variable(0, dtype=tf.int64, trainable=False)
    refine_epoch_var = tf.Variable(0, dtype=tf.int64, trainable=False)

    checkpoint_manager = None
    if ACTIVE_METHOD_DIR is not None:
        checkpoint = tf.train.Checkpoint(
            student=student,
            scrub_optimizer=optimizer,
            refine_optimizer=optimizer_refine,
            scrub_epoch=scrub_epoch_var,
            refine_epoch=refine_epoch_var,
        )
        checkpoint_manager = tf.train.CheckpointManager(
            checkpoint,
            str(ACTIVE_METHOD_DIR / "custom_checkpoint"),
            max_to_keep=2,
        )
        if checkpoint_manager.latest_checkpoint:
            checkpoint.restore(checkpoint_manager.latest_checkpoint).expect_partial()
            print(
                "Resumed ScrubRef checkpoint:",
                checkpoint_manager.latest_checkpoint,
                "scrub_epoch=", int(scrub_epoch_var.numpy()),
                "refine_epoch=", int(refine_epoch_var.numpy()),
            )

    student_features = build_feature_model(student, bundle.num_classes)
    teacher_features = build_feature_model(teacher, bundle.num_classes)

    retain_ds = make_dataset(
        split.retain_train_items,
        split.retain_train_labels,
        bundle,
        cfg.batch_size,
        True,
        True,
        True,
        seed,
    )
    proxy_bundle = DataBundle(
        name="proxy",
        train_items=proxies,
        train_labels=proxy_labels,
        test_items=proxies,
        test_labels=proxy_labels,
        class_names=bundle.class_names,
        image_size=bundle.image_size,
        path_mode=False,
    )
    proxy_ds = make_dataset(proxies, proxy_labels, proxy_bundle, cfg.batch_size, True, False, True, seed + 1)
    retain_iterator = iter(retain_ds)
    proxy_iterator = iter(proxy_ds)
    steps_per_epoch = max(1, math.ceil(len(split.retain_train_items) / cfg.batch_size))

    @tf.function(reduce_retracing=True)
    def train_step(retain_images: tf.Tensor, retain_labels: tf.Tensor, proxy_images: tf.Tensor) -> Tuple[tf.Tensor, ...]:
        mixed_images, mixed_labels = mixup_batch(retain_images, retain_labels, bundle.num_classes, cfg.mixup_alpha)
        with tf.GradientTape() as tape:
            student_retain_output = student(mixed_images, training=True)
            student_proxy_output = student(proxy_images, training=True)
            teacher_retain_output = teacher(mixed_images, training=False)

            retain_ce = categorical_ce_from_output(mixed_labels, student_retain_output, output_is_logits)
            teacher_masked = masked_distribution(
                teacher_retain_output, output_is_logits, forget_classes, cfg.kd_temperature
            )
            student_masked = masked_distribution(
                student_retain_output, output_is_logits, forget_classes, cfg.kd_temperature
            )
            retain_kd = categorical_kl(teacher_masked, student_masked) * (cfg.kd_temperature ** 2)

            student_proxy_probs = output_to_probs(student_proxy_output, output_is_logits)
            mass = forgot_mass(student_proxy_probs, forget_classes)
            forget_output_loss = tf.reduce_mean(-tf.math.log(tf.clip_by_value(1.0 - mass, 1e-7, 1.0)))
            entropy = retained_entropy(student_proxy_probs, forget_classes)
            forget_entropy_loss = -tf.reduce_mean(entropy)

            student_retain_features = tf.reshape(
                tf.cast(student_features(mixed_images, training=True), tf.float32),
                [tf.shape(mixed_images)[0], -1],
            )
            teacher_retain_features = tf.reshape(
                tf.cast(teacher_features(mixed_images, training=False), tf.float32),
                [tf.shape(mixed_images)[0], -1],
            )
            student_retain_features = tf.math.l2_normalize(student_retain_features, axis=1)
            teacher_retain_features = tf.math.l2_normalize(teacher_retain_features, axis=1)
            retain_feature_loss = tf.reduce_mean(tf.square(student_retain_features - teacher_retain_features))

            forget_feature_loss = tf.constant(0.0, tf.float32)
            if representation_aware:
                student_proxy_features = tf.reshape(
                    tf.cast(student_features(proxy_images, training=True), tf.float32),
                    [tf.shape(proxy_images)[0], -1],
                )
                student_proxy_features = tf.math.l2_normalize(student_proxy_features, axis=1)
                retained_centroid = tf.math.l2_normalize(
                    tf.reduce_mean(teacher_retain_features, axis=0, keepdims=True), axis=1
                )
                cosine = tf.reduce_sum(student_proxy_features * retained_centroid, axis=1)
                forget_feature_loss = tf.reduce_mean(1.0 - cosine)

            total_loss = (
                cfg.retain_ce_weight * retain_ce
                + cfg.retain_kd_weight * retain_kd
                + cfg.retain_feature_weight * retain_feature_loss
                + cfg.forget_output_weight * forget_output_loss
                + cfg.forget_entropy_weight * forget_entropy_loss
                + cfg.forget_feature_weight * forget_feature_loss
            )

        gradients = tape.gradient(total_loss, student.trainable_variables)
        gradients, _ = tf.clip_by_global_norm(gradients, cfg.gradient_clip_norm)
        optimizer.apply_gradients(zip(gradients, student.trainable_variables))
        return (
            total_loss,
            retain_ce,
            retain_kd,
            retain_feature_loss,
            forget_output_loss,
            forget_entropy_loss,
            forget_feature_loss,
            gradient_global_norm(gradients),
        )

    probe_images, _ = next(proxy_iterator)
    with tf.GradientTape() as audit_tape:
        audit_output = student(probe_images, training=True)
        audit_probs = output_to_probs(audit_output, output_is_logits)
        audit_mass = forgot_mass(audit_probs, forget_classes)
        audit_loss = tf.reduce_mean(-tf.math.log(tf.clip_by_value(1.0 - audit_mass, 1e-7, 1.0)))
    audit_gradients = audit_tape.gradient(audit_loss, student.trainable_variables)
    proxy_gradient_norm = float(gradient_global_norm(audit_gradients).numpy())
    if not np.isfinite(proxy_gradient_norm) or proxy_gradient_norm <= 0.0:
        raise RuntimeError("The proxy forget branch has zero or invalid gradient.")

    last_scrub_loss = float("nan")
    for epoch in range(int(scrub_epoch_var.numpy()), cfg.scrub_epochs):
        epoch_values: List[np.ndarray] = []
        epoch_started = time.perf_counter()
        for _ in range(steps_per_epoch):
            retain_images, retain_labels = next(retain_iterator)
            proxy_images, _ = next(proxy_iterator)
            values = train_step(retain_images, retain_labels, proxy_images)
            epoch_values.append(np.asarray([float(v.numpy()) for v in values], dtype=np.float64))
        means = np.mean(epoch_values, axis=0)
        last_scrub_loss = float(means[0])
        scrub_epoch_var.assign(epoch + 1)
        if checkpoint_manager is not None:
            checkpoint_manager.save(checkpoint_number=epoch + 1)
        print({
            "phase": "scrub",
            "variant": "representation" if representation_aware else "head",
            "epoch": epoch + 1,
            "epochs": cfg.scrub_epochs,
            "loss": last_scrub_loss,
            "forget_output": float(means[4]),
            "forget_feature": float(means[6]),
            "elapsed_seconds": time.perf_counter() - epoch_started,
        })

    if cfg.enable_main_method_logit_surgery:
        apply_logit_surgery(student, bundle.num_classes, forget_classes, cfg.logit_surgery_shift)

    set_tail_trainable(student, bundle.num_classes, tail_layers)
    retain_refine_ds = make_dataset(
        split.retain_train_items,
        split.retain_train_labels,
        bundle,
        cfg.batch_size,
        True,
        True,
        True,
        seed + 2,
    )
    refine_iterator = iter(retain_refine_ds)
    refine_steps = max(1, math.ceil(len(split.retain_train_items) / cfg.batch_size))

    @tf.function(reduce_retracing=True)
    def refine_step(images: tf.Tensor, labels: tf.Tensor) -> tf.Tensor:
        mixed_images, mixed_labels = mixup_batch(images, labels, bundle.num_classes, cfg.mixup_alpha)
        with tf.GradientTape() as tape:
            student_output = student(mixed_images, training=True)
            teacher_output = teacher(mixed_images, training=False)
            ce = categorical_ce_from_output(mixed_labels, student_output, output_is_logits)
            teacher_masked = masked_distribution(teacher_output, output_is_logits, forget_classes, cfg.kd_temperature)
            student_masked = masked_distribution(student_output, output_is_logits, forget_classes, cfg.kd_temperature)
            kd = categorical_kl(teacher_masked, student_masked) * (cfg.kd_temperature ** 2)
            loss = ce + cfg.retain_kd_weight * kd
        gradients = tape.gradient(loss, student.trainable_variables)
        gradients, _ = tf.clip_by_global_norm(gradients, cfg.gradient_clip_norm)
        optimizer_refine.apply_gradients(zip(gradients, student.trainable_variables))
        return loss

    for epoch in range(int(refine_epoch_var.numpy()), cfg.refine_epochs):
        epoch_started = time.perf_counter()
        losses = []
        for _ in range(refine_steps):
            images, labels = next(refine_iterator)
            losses.append(float(refine_step(images, labels).numpy()))
        refine_epoch_var.assign(epoch + 1)
        if checkpoint_manager is not None:
            checkpoint_manager.save(checkpoint_number=cfg.scrub_epochs + epoch + 1)
        print({
            "phase": "refine",
            "epoch": epoch + 1,
            "epochs": cfg.refine_epochs,
            "loss": float(np.mean(losses)),
            "elapsed_seconds": time.perf_counter() - epoch_started,
        })

    if cfg.enable_main_method_clamp:
        student = confidence_clamp(
            student,
            proxies,
            proxy_labels,
            bundle,
            forget_classes,
            output_is_logits,
            cfg,
            seed,
        )

    metadata = {
        "proxy_gradient_norm": proxy_gradient_norm,
        "trainable_layer_count": float(len(trainable_names)),
        "representation_aware": float(representation_aware),
        "final_scrub_loss": last_scrub_loss,
        "resumed_scrub_epoch": float(scrub_epoch_var.numpy()),
        "resumed_refine_epoch": float(refine_epoch_var.numpy()),
    }
    return student, metadata


def confidence_clamp(
    model: tf.keras.Model,
    proxies: np.ndarray,
    proxy_labels: np.ndarray,
    bundle: DataBundle,
    forget_classes: Sequence[int],
    output_is_logits: bool,
    cfg: ExperimentConfig,
    seed: int,
) -> tf.keras.Model:
    set_tail_trainable(model, bundle.num_classes, cfg.head_tail_layers)
    optimizer = tf.keras.optimizers.Adam(cfg.refine_lr * 0.1)
    proxy_bundle = DataBundle(
        name="proxy",
        train_items=proxies,
        train_labels=proxy_labels,
        test_items=proxies,
        test_labels=proxy_labels,
        class_names=bundle.class_names,
        image_size=bundle.image_size,
        path_mode=False,
    )
    ds = make_dataset(proxies, proxy_labels, proxy_bundle, cfg.batch_size, True, False, False, seed)
    for _ in range(cfg.clamp_epochs):
        for images, _ in ds:
            with tf.GradientTape() as tape:
                probs = output_to_probs(model(images, training=True), output_is_logits)
                mass = forgot_mass(probs, forget_classes)
                loss = tf.reduce_mean(tf.nn.relu(mass - cfg.clamp_threshold) ** 2)
            gradients = tape.gradient(loss, model.trainable_variables)
            gradients, _ = tf.clip_by_global_norm(gradients, cfg.gradient_clip_norm)
            optimizer.apply_gradients(zip(gradients, model.trainable_variables))
    return model


### Baselines with resumable training

This section implements the baseline methods used to compare ScrubRef-Rep with alternative class-unlearning strategies.

- **Logit Clamp**: Directly suppresses forgotten-class classifier weights and shifts their biases downward without additional training.
- **Head Reset**: Reinitializes the classifier head and retrains it using retained data only.
- **Retain Fine-tuning**: Fine-tunes the upper layers of the original model on retained samples.
- **UNSIR-style impair-and-repair**: First trains on retained samples mixed with error-maximizing synthetic noise, then repairs utility through retained-data training.
- **Bad Teacher**: Preserves retained-class behaviour through a competent teacher while encouraging uniform predictions on forgotten samples.
- **SCRUB-style baseline**: Uses retained supervision and knowledge distillation while maximizing a bounded divergence from the teacher on forgotten samples.
- **SSD**: Estimates diagonal Fisher information on retained and forgotten data, then selectively dampens parameters that are more important for forgotten classes.
- **Retraining**: Reinitializes the model and trains it from scratch using retained data only, serving as the reference unlearning baseline.

The Bad Teacher and SCRUB-style baselines use `tf.train.CheckpointManager` to save the student model, optimizer state, and completed epoch after each epoch. When a checkpoint is available, training resumes from the latest saved state.


In [ ]:
def logit_clamp_baseline(
    original: tf.keras.Model,
    bundle: DataBundle,
    forget_classes: Sequence[int],
    shift: float,
) -> tf.keras.Model:
    model = clone_with_weights(original)
    classifier = classifier_layer(model, bundle.num_classes)
    weights = classifier.get_weights()
    kernel = weights[0].copy()
    kernel[:, list(forget_classes)] = 0.0
    if len(weights) == 2:
        bias = weights[1].copy()
        bias[list(forget_classes)] -= shift
        classifier.set_weights([kernel, bias])
    else:
        classifier.set_weights([kernel])
    return model


def head_reset_baseline(
    original: tf.keras.Model,
    split: UnlearningSplit,
    bundle: DataBundle,
    output_is_logits: bool,
    cfg: ExperimentConfig,
    seed: int,
) -> tf.keras.Model:
    model = clone_with_weights(original)
    classifier = classifier_layer(model, bundle.num_classes)
    weights = classifier.get_weights()
    kernel = tf.keras.initializers.GlorotUniform(seed=seed)(weights[0].shape).numpy()
    if len(weights) == 2:
        bias = np.zeros_like(weights[1])
        classifier.set_weights([kernel, bias])
    else:
        classifier.set_weights([kernel])
    return train_supervised(
        model,
        split.retain_train_items,
        split.retain_train_labels,
        bundle,
        output_is_logits,
        cfg.baseline_epochs,
        cfg.batch_size,
        cfg.baseline_lr,
        seed,
        validation=(split.retain_val_items, split.retain_val_labels),
        train_all=False,
        tail_layers=0,
    )


def retain_finetune_baseline(
    original: tf.keras.Model,
    split: UnlearningSplit,
    bundle: DataBundle,
    output_is_logits: bool,
    cfg: ExperimentConfig,
    seed: int,
) -> tf.keras.Model:
    model = clone_with_weights(original)
    return train_supervised(
        model,
        split.retain_train_items,
        split.retain_train_labels,
        bundle,
        output_is_logits,
        cfg.baseline_epochs,
        cfg.batch_size,
        cfg.baseline_lr,
        seed,
        validation=(split.retain_val_items, split.retain_val_labels),
        train_all=False,
        tail_layers=cfg.representation_tail_layers,
    )


def unsir_baseline(
    original: tf.keras.Model,
    split: UnlearningSplit,
    bundle: DataBundle,
    forget_classes: Sequence[int],
    output_is_logits: bool,
    cfg: ExperimentConfig,
    seed: int,
) -> tf.keras.Model:
    """GPU-safe UNSIR-style impair-and-repair baseline."""
    noise_count = min(int(cfg.proxy_count), 64)
    noise, noise_labels = generate_unsir_error_maximizing_noise(
        original,
        bundle,
        forget_classes,
        output_is_logits,
        count=noise_count,
        steps=int(cfg.unsir_noise_steps),
        seed=seed,
    )
    model = clone_with_weights(original)
    retain_images, retain_labels = materialize_items(
        split.retain_train_items,
        split.retain_train_labels,
        bundle,
        limit=min(len(noise), len(split.retain_train_items)),
        seed=seed,
    )
    impair_images = np.concatenate([retain_images, noise], axis=0)
    impair_labels = np.concatenate([retain_labels, noise_labels], axis=0)
    array_bundle = DataBundle(
        name=bundle.name,
        train_items=impair_images,
        train_labels=impair_labels,
        test_items=impair_images,
        test_labels=impair_labels,
        class_names=bundle.class_names,
        image_size=bundle.image_size,
        path_mode=False,
    )
    model = train_supervised(
        model,
        impair_images,
        impair_labels,
        array_bundle,
        output_is_logits,
        epochs=1,
        batch_size=cfg.batch_size,
        learning_rate=cfg.unsir_impair_lr,
        seed=seed,
        train_all=True,
    )
    model = train_supervised(
        model,
        retain_images,
        retain_labels,
        array_bundle,
        output_is_logits,
        epochs=max(1, cfg.baseline_epochs // 2),
        batch_size=cfg.batch_size,
        learning_rate=cfg.unsir_repair_lr,
        seed=seed + 1,
        train_all=True,
    )
    del noise, noise_labels, retain_images, retain_labels, impair_images, impair_labels
    gc.collect()
    return model



def bad_teacher_baseline(
    original: tf.keras.Model,
    split: UnlearningSplit,
    bundle: DataBundle,
    output_is_logits: bool,
    cfg: ExperimentConfig,
    seed: int,
) -> tf.keras.Model:
    """Memory-safe Bad Teacher-style baseline using a uniform incompetent teacher."""
    student = clone_with_weights(original)
    competent = original
    competent.trainable = False
    set_tail_trainable(
        student,
        bundle.num_classes,
        max(4, min(int(cfg.representation_tail_layers), 8)),
    )
    optimizer = tf.keras.optimizers.Adam(min(float(cfg.baseline_lr), 2e-4))
    epoch_var = tf.Variable(0, dtype=tf.int64, trainable=False)
    manager = None

    if ACTIVE_METHOD_DIR is not None:
        checkpoint = tf.train.Checkpoint(student=student, optimizer=optimizer, epoch=epoch_var)
        manager = tf.train.CheckpointManager(
            checkpoint,
            str(ACTIVE_METHOD_DIR / "custom_checkpoint"),
            max_to_keep=2,
        )
        if manager.latest_checkpoint:
            checkpoint.restore(manager.latest_checkpoint).expect_partial()
            print("Resumed Bad Teacher at epoch", int(epoch_var.numpy()))

    retain_ds = make_dataset(
        split.retain_train_items,
        split.retain_train_labels,
        bundle,
        cfg.batch_size,
        True,
        True,
        True,
        seed,
    )
    forget_ds = make_dataset(
        split.forget_train_items,
        split.forget_train_labels,
        bundle,
        cfg.batch_size,
        True,
        True,
        True,
        seed + 1,
    )
    retain_it, forget_it = iter(retain_ds), iter(forget_ds)
    steps = max(1, math.ceil(len(split.retain_train_items) / cfg.batch_size))

    @tf.function(reduce_retracing=True)
    def step(
        retain_images: tf.Tensor,
        retain_labels: tf.Tensor,
        forget_images: tf.Tensor,
    ) -> tf.Tensor:
        with tf.GradientTape() as tape:
            student_retain = student(retain_images, training=True)
            student_forget = student(forget_images, training=True)
            competent_retain = competent(retain_images, training=False)
            retain_probs = output_to_probs(
                competent_retain,
                output_is_logits,
                cfg.kd_temperature,
            )
            student_retain_probs = output_to_probs(
                student_retain,
                output_is_logits,
                cfg.kd_temperature,
            )
            student_forget_probs = output_to_probs(
                student_forget,
                output_is_logits,
                cfg.kd_temperature,
            )
            incompetent_probs = tf.fill(
                tf.shape(student_forget_probs),
                tf.cast(1.0 / bundle.num_classes, student_forget_probs.dtype),
            )
            ce = tf.reduce_mean(
                tf.keras.losses.sparse_categorical_crossentropy(
                    retain_labels,
                    student_retain,
                    from_logits=output_is_logits,
                )
            )
            kd_retain = categorical_kl(retain_probs, student_retain_probs)
            kd_forget = categorical_kl(incompetent_probs, student_forget_probs)
            loss = ce + 0.5 * kd_retain + 0.5 * kd_forget
        gradients = tape.gradient(loss, student.trainable_variables)
        gradients, _ = tf.clip_by_global_norm(gradients, 1.0)
        optimizer.apply_gradients(zip(gradients, student.trainable_variables))
        return loss

    for epoch in range(int(epoch_var.numpy()), cfg.baseline_epochs):
        started = time.perf_counter()
        losses: List[float] = []
        for _ in range(steps):
            retain_images, retain_labels = next(retain_it)
            forget_images, _ = next(forget_it)
            losses.append(float(step(retain_images, retain_labels, forget_images).numpy()))
        epoch_var.assign(epoch + 1)
        if manager is not None:
            manager.save(checkpoint_number=epoch + 1)
        print(
            f"Bad Teacher epoch {epoch + 1}/{cfg.baseline_epochs}: "
            f"loss={np.mean(losses):.6f}, elapsed={time.perf_counter() - started:.1f}s"
        )

    del competent, retain_ds, forget_ds, retain_it, forget_it, step
    gc.collect()
    return student



def scrub_baseline(
    original: tf.keras.Model,
    split: UnlearningSplit,
    bundle: DataBundle,
    output_is_logits: bool,
    cfg: ExperimentConfig,
    seed: int,
) -> tf.keras.Model:
    """Stable, bounded SCRUB-style baseline that avoids catastrophic collapse."""
    student = clone_with_weights(original)
    teacher = original
    teacher.trainable = False
    set_tail_trainable(
        student,
        bundle.num_classes,
        max(4, min(int(cfg.representation_tail_layers), 8)),
    )
    optimizer = tf.keras.optimizers.Adam(min(float(cfg.baseline_lr), 1e-4))
    epoch_var = tf.Variable(0, dtype=tf.int64, trainable=False)
    manager = None

    if ACTIVE_METHOD_DIR is not None:
        checkpoint = tf.train.Checkpoint(student=student, optimizer=optimizer, epoch=epoch_var)
        manager = tf.train.CheckpointManager(
            checkpoint,
            str(ACTIVE_METHOD_DIR / "custom_checkpoint_v6"),
            max_to_keep=2,
        )
        if manager.latest_checkpoint:
            checkpoint.restore(manager.latest_checkpoint).expect_partial()
            print("Resumed stable SCRUB at epoch", int(epoch_var.numpy()))

    retain_ds = make_dataset(
        split.retain_train_items,
        split.retain_train_labels,
        bundle,
        cfg.batch_size,
        True,
        True,
        True,
        seed,
    )
    forget_ds = make_dataset(
        split.forget_train_items,
        split.forget_train_labels,
        bundle,
        cfg.batch_size,
        True,
        True,
        True,
        seed + 1,
    )
    retain_it, forget_it = iter(retain_ds), iter(forget_ds)
    steps = max(1, math.ceil(len(split.retain_train_items) / cfg.batch_size))

    @tf.function(reduce_retracing=True)
    def combined_step(
        retain_images: tf.Tensor,
        retain_labels: tf.Tensor,
        forget_images: tf.Tensor,
    ) -> Tuple[tf.Tensor, tf.Tensor, tf.Tensor]:
        with tf.GradientTape() as tape:
            student_retain = student(retain_images, training=True)
            student_forget = student(forget_images, training=True)
            teacher_retain = teacher(retain_images, training=False)
            teacher_forget = teacher(forget_images, training=False)

            ce = tf.reduce_mean(
                tf.keras.losses.sparse_categorical_crossentropy(
                    retain_labels,
                    student_retain,
                    from_logits=output_is_logits,
                )
            )
            retain_kd = categorical_kl(
                output_to_probs(teacher_retain, output_is_logits, cfg.kd_temperature),
                output_to_probs(student_retain, output_is_logits, cfg.kd_temperature),
            )
            forget_divergence = categorical_kl(
                output_to_probs(teacher_forget, output_is_logits, cfg.kd_temperature),
                output_to_probs(student_forget, output_is_logits, cfg.kd_temperature),
            )
            bounded_forget = tf.math.tanh(forget_divergence)
            loss = ce + 0.5 * retain_kd - 0.05 * bounded_forget

        gradients = tape.gradient(loss, student.trainable_variables)
        gradients, _ = tf.clip_by_global_norm(gradients, 1.0)
        optimizer.apply_gradients(zip(gradients, student.trainable_variables))
        return loss, retain_kd, forget_divergence

    for epoch in range(int(epoch_var.numpy()), cfg.baseline_epochs):
        started = time.perf_counter()
        losses: List[float] = []
        retain_kds: List[float] = []
        forget_divergences: List[float] = []
        for _ in range(steps):
            retain_images, retain_labels = next(retain_it)
            forget_images, _ = next(forget_it)
            loss, retain_kd, forget_divergence = combined_step(
                retain_images,
                retain_labels,
                forget_images,
            )
            losses.append(float(loss.numpy()))
            retain_kds.append(float(retain_kd.numpy()))
            forget_divergences.append(float(forget_divergence.numpy()))

        epoch_var.assign(epoch + 1)
        if manager is not None:
            manager.save(checkpoint_number=epoch + 1)
        print(
            f"Stable SCRUB epoch {epoch + 1}/{cfg.baseline_epochs}: "
            f"loss={np.mean(losses):.6f}, "
            f"retain_kd={np.mean(retain_kds):.6f}, "
            f"forget_divergence={np.mean(forget_divergences):.6f}, "
            f"elapsed={time.perf_counter() - started:.1f}s"
        )

    del teacher, retain_ds, forget_ds, retain_it, forget_it, combined_step
    gc.collect()
    return student



def estimate_fisher_diagonal(
    model: tf.keras.Model,
    items: np.ndarray,
    labels: np.ndarray,
    bundle: DataBundle,
    output_is_logits: bool,
    batch_size: int,
    max_batches: int,
    seed: int,
) -> List[tf.Tensor]:
    set_all_trainable(model, True, freeze_batch_norm=True)
    variables = model.trainable_variables
    fisher = [tf.zeros_like(variable, dtype=tf.float32) for variable in variables]
    ds = make_dataset(items, labels, bundle, batch_size, True, False, False, seed)
    batch_count = 0
    for images, targets in ds.take(max_batches):
        with tf.GradientTape() as tape:
            output = model(images, training=False)
            loss = tf.reduce_mean(tf.keras.losses.sparse_categorical_crossentropy(
                targets, output, from_logits=output_is_logits
            ))
        gradients = tape.gradient(loss, variables)
        fisher = [
            accumulated + (tf.square(tf.cast(gradient, tf.float32)) if gradient is not None else 0.0)
            for accumulated, gradient in zip(fisher, gradients)
        ]
        batch_count += 1
    return [value / max(batch_count, 1) for value in fisher]


def ssd_baseline(
    original: tf.keras.Model,
    split: UnlearningSplit,
    bundle: DataBundle,
    output_is_logits: bool,
    cfg: ExperimentConfig,
    seed: int,
) -> tf.keras.Model:
    model = clone_with_weights(original)
    retain_fisher = estimate_fisher_diagonal(
        model,
        split.retain_train_items,
        split.retain_train_labels,
        bundle,
        output_is_logits,
        cfg.batch_size,
        cfg.ssd_fisher_batches,
        seed,
    )
    forget_fisher = estimate_fisher_diagonal(
        model,
        split.forget_train_items,
        split.forget_train_labels,
        bundle,
        output_is_logits,
        cfg.batch_size,
        cfg.ssd_fisher_batches,
        seed + 1,
    )
    epsilon = 1e-12
    for variable, retain_value, forget_value in zip(model.trainable_variables, retain_fisher, forget_fisher):
        ratio = forget_value / (retain_value + epsilon)
        selected = ratio > cfg.ssd_selection_weighting
        dampening = tf.minimum(
            cfg.ssd_dampening_constant * retain_value / (forget_value + epsilon),
            tf.ones_like(forget_value),
        )
        factor = tf.where(selected, dampening, tf.ones_like(dampening))
        variable.assign(tf.cast(variable, tf.float32) * factor)
    return model


def retrain_baseline(
    original: tf.keras.Model,
    split: UnlearningSplit,
    bundle: DataBundle,
    output_is_logits: bool,
    cfg: ExperimentConfig,
    seed: int,
) -> tf.keras.Model:
    initial_path = cfg.initial_weights_paths.get(cfg.dataset, "")
    model = fresh_clone(original, initial_path)
    model = train_supervised(
        model,
        split.retain_train_items,
        split.retain_train_labels,
        bundle,
        output_is_logits,
        cfg.retrain_epochs,
        cfg.batch_size,
        cfg.retrain_lr,
        seed,
        validation=(split.retain_val_items, split.retain_val_labels),
        train_all=True,
    )
    model = train_supervised(
        model,
        split.retain_train_items,
        split.retain_train_labels,
        bundle,
        output_is_logits,
        V7_RETRAIN_EXTRA_EPOCHS,
        cfg.batch_size,
        V7_RETRAIN_PHASE2_LR,
        seed + 1000,
        validation=(split.retain_val_items, split.retain_val_labels),
        train_all=True,
    )
    return model


### Staged experiment runner and checkpoint manifest


This section provides the experiment orchestration layer for reproducible, multi-seed class-unlearning evaluation.

Each seed is prepared independently by creating a structured run directory, loading or training the shared original model, generating class-conditioned proxies, and saving the resolved experiment configuration and original-model metadata. Method-specific outputs are then organized into separate model, metric, feature, proxy, and checkpoint-stage directories.

The runner supports the following workflow:

1. **Seed preparation**: `prepare_seed` initializes the run directory, preserves the shared original model, generates or reuses proxy samples, and records preparation status.
2. **Method dispatch**: `build_method` maps each configured method name to its corresponding ScrubRef variant or baseline implementation.
3. **Resumable execution**: `run_one_method` detects saved models and partial checkpoint artifacts, resumes compatible training procedures when possible, and avoids recomputing completed methods unless `force=True`.
4. **Atomic artifact writing**: Model outputs, utility metrics, membership-inference metrics, runtime measurements, per-class accuracy, and metadata are saved as separate artifacts to reduce corruption risk during long experiments.
5. **Artifact validation**: SCRUB outputs are checked for severe utility collapse before they are treated as complete. Invalid artifacts are archived with a UTC timestamp and rebuilt.
6. **Evaluation and parameter auditing**: Every saved method model is evaluated for utility, membership-inference privacy, per-class accuracy, GPU-memory usage, runtime, and relative parameter change from the original model.
7. **Status dashboard**: Per-seed JSON status files create a lightweight checkpoint manifest that reports whether each preparation or method stage is complete, running, failed, partially checkpointed, or not started.

The final artifact layout makes interrupted experiments recoverable and keeps training, evaluation, checkpoints, and migration records separated for each seed and method.

In [ ]:
def method_model_path(run_dir: Path, method: str) -> Path:
    if method == "original":
        return run_dir.parent / "shared" / "original.keras"
    return run_dir / "models" / f"{method}.keras"


def method_stage_dir(run_dir: Path, method: str) -> Path:
    return run_dir / "stages" / method


def method_metric_path(run_dir: Path, method: str, metric_group: str) -> Path:
    return run_dir / "metrics" / f"{method}_{metric_group}.csv"


def load_saved_model(path: Path) -> tf.keras.Model:
    return tf.keras.models.load_model(
        path,
        custom_objects=custom_objects(),
        compile=False,
        safe_mode=False,
    )


def load_original_metadata(cfg: ExperimentConfig) -> Dict[str, object]:
    path = experiment_dir(cfg) / "shared" / "original_metadata.json"
    if not path.exists():
        raise FileNotFoundError("Run a Stage 3 seed-preparation cell first.")
    return json.loads(path.read_text(encoding="utf-8"))


In [ ]:
def build_method(
    method: str,
    original: tf.keras.Model,
    split: UnlearningSplit,
    bundle: DataBundle,
    proxies: np.ndarray,
    proxy_labels: np.ndarray,
    forget_classes: Sequence[int],
    output_is_logits: bool,
    cfg: ExperimentConfig,
    seed: int,
) -> Tuple[tf.keras.Model, Dict[str, float]]:
    metadata: Dict[str, float] = {}
    if method == "original":
        return original, metadata
    if method == "retrained":
        return retrain_baseline(original, split, bundle, output_is_logits, cfg, seed), metadata
    if method == "logit_clamp":
        return logit_clamp_baseline(original, bundle, forget_classes, shift=20.0), metadata
    if method == "head_reset":
        return head_reset_baseline(original, split, bundle, output_is_logits, cfg, seed), metadata
    if method == "retain_finetune":
        return retain_finetune_baseline(original, split, bundle, output_is_logits, cfg, seed), metadata
    if method == "scrubref_head":
        return scrubref_train(
            original, split, bundle, proxies, proxy_labels, forget_classes, output_is_logits, cfg, seed, False
        )
    if method == "scrubref_rep":
        return scrubref_train(
            original, split, bundle, proxies, proxy_labels, forget_classes, output_is_logits, cfg, seed, True
        )
    if method == "unsir":
        return unsir_baseline(original, split, bundle, forget_classes, output_is_logits, cfg, seed), metadata
    if method == "bad_teacher":
        return bad_teacher_baseline(original, split, bundle, output_is_logits, cfg, seed), metadata
    if method == "scrub":
        return scrub_baseline(original, split, bundle, output_is_logits, cfg, seed), metadata
    if method == "ssd":
        return ssd_baseline(original, split, bundle, output_is_logits, cfg, seed), metadata
    raise ValueError(f"Unknown method: {method}")

from src.utils import atomic_write_json, set_stage_status, atomic_write_dataframe
def prepare_seed(cfg: ExperimentConfig, seed: int, force_proxy: bool = False) -> Dict[str, Path]:
    reset_tensorflow(seed)
    run_dir = seed_dir(cfg, seed)
    for subdir in ("models", "metrics", "status", "stages", "features", "figures", "proxies"):
        (run_dir / subdir).mkdir(parents=True, exist_ok=True)
    atomic_write_json(run_dir / "config.json", asdict(cfg))
    set_stage_status(cfg, seed, "prepare", "running")

    bundle = load_dataset(cfg)
    if not bundle.path_mode and bundle.train_items.dtype != np.uint8:
        raise RuntimeError("CIFAR arrays must remain uint8 in RAM.")

    original, output_is_logits, original_source = load_or_train_original(cfg, bundle, seed, run_dir)
    shared_dir = experiment_dir(cfg) / "shared"
    shared_dir.mkdir(parents=True, exist_ok=True)
    shared_original = shared_dir / "original.keras"
    if not shared_original.exists():
        source = Path(original_source)
        if source.suffix == ".keras":
            shutil.copy2(source, shared_original)
        else:
            original.save(shared_original)
    atomic_write_json(shared_dir / "original_metadata.json", {
        "source": original_source,
        "output_is_logits": bool(output_is_logits),
        "dataset": cfg.dataset,
        "config_fingerprint": config_fingerprint(cfg),
    })

    proxy_path = run_dir / "proxies" / "proxies.npz"
    if force_proxy and proxy_path.exists():
        proxy_path.unlink()
    proxies, proxy_labels, proxy_metadata = generate_class_proxies(
        original,
        bundle,
        cfg.forget_classes_by_dataset[cfg.dataset],
        output_is_logits,
        cfg,
        seed,
        proxy_path,
    )
    atomic_write_json(run_dir / "proxies" / "proxy_metadata.json", proxy_metadata)
    set_stage_status(
        cfg,
        seed,
        "prepare",
        "complete",
        proxy_count=len(proxy_labels),
        output_is_logits=bool(output_is_logits),
    )
    del original, bundle, proxies, proxy_labels
    release_memory(clear_session=True)
    return {"run_dir": run_dir, "proxy_path": proxy_path, "original_path": shared_original}


def prepare_all_seeds(cfg: ExperimentConfig, seeds: Optional[Sequence[int]] = None) -> None:
    selected = tuple(seeds or cfg.seeds)
    for seed in selected:
        print("\n" + "=" * 90)
        print(f"Preparing {cfg.dataset}, seed={seed}")
        print("=" * 90)
        prepare_seed(cfg, seed)


def parameter_change_statistics_from_weights(
    original_weights: Sequence[np.ndarray],
    changed_weights: Sequence[np.ndarray],
    tolerance: float = 1e-7,
) -> Dict[str, float]:
    if len(original_weights) != len(changed_weights):
        return {
            "changed_parameter_fraction": float("nan"),
            "relative_l2_parameter_change": float("nan"),
        }
    total = 0
    changed_count = 0
    squared_delta = 0.0
    squared_original = 0.0
    for before, after in zip(original_weights, changed_weights):
        before64 = np.asarray(before, dtype=np.float64)
        after64 = np.asarray(after, dtype=np.float64)
        delta = after64 - before64
        total += delta.size
        changed_count += int(np.count_nonzero(np.abs(delta) > tolerance))
        squared_delta += float(np.sum(delta ** 2))
        squared_original += float(np.sum(before64 ** 2))
    return {
        "changed_parameter_fraction": float(changed_count / max(total, 1)),
        "relative_l2_parameter_change": float(
            math.sqrt(squared_delta) / max(math.sqrt(squared_original), 1e-12)
        ),
    }


def method_artifact_quality_issue(
    method: str,
    utility_path: Path,
) -> Optional[str]:
    if method != "scrub" or not utility_path.exists():
        return None
    try:
        utility = pd.read_csv(utility_path)
        if utility.empty:
            return "empty utility file"
        retained = float(utility.iloc[0]["retained_accuracy"])
        overall = float(utility.iloc[0]["overall_accuracy"])
        if retained < 0.20 or overall < 0.20:
            return (
                "collapsed SCRUB artifact detected "
                f"(retained_accuracy={retained:.4f}, overall_accuracy={overall:.4f})"
            )
    except Exception as exc:
        return f"unreadable SCRUB utility file: {exc!r}"
    return None


def archive_invalid_method_artifacts(
    run_dir: Path,
    method: str,
) -> Path:
    timestamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
    archive_dir = run_dir / "invalidated_v7" / f"{method}_{timestamp}"
    archive_dir.mkdir(parents=True, exist_ok=True)

    candidates = [
        method_model_path(run_dir, method),
        run_dir / "metrics" / f"{method}_utility.csv",
        run_dir / "metrics" / f"{method}_mia.csv",
        run_dir / "metrics" / f"{method}_runtime.csv",
        run_dir / "metrics" / f"{method}_per_class.csv",
        run_dir / "metrics" / f"{method}_metadata.json",
        run_dir / "features" / f"{method}.npz",
        run_dir / "features" / f"{method}_probs.npz",
    ]
    for source in candidates:
        if source.exists() and method != "original":
            shutil.move(str(source), str(archive_dir / source.name))

    stage_dir = method_stage_dir(run_dir, method)
    if stage_dir.exists():
        shutil.move(str(stage_dir), str(archive_dir / stage_dir.name))
    return archive_dir


def v6_resume_report(
    cfg: ExperimentConfig,
    seed: int,
) -> pd.DataFrame:
    run_dir = seed_dir(cfg, seed)
    rows: List[Dict[str, object]] = []
    for method in cfg.core_methods:
        model_path = method_model_path(run_dir, method)
        utility_path = method_metric_path(run_dir, method, "utility")
        mia_path = method_metric_path(run_dir, method, "mia")
        runtime_path = method_metric_path(run_dir, method, "runtime")
        class_path = method_metric_path(run_dir, method, "per_class")
        issue = method_artifact_quality_issue(method, utility_path)
        complete = all(
            path.exists()
            for path in (model_path, utility_path, mia_path, runtime_path, class_path)
        ) and issue is None
        if complete:
            state = "complete"
        elif model_path.exists() and issue is None:
            state = "model_saved_evaluation_pending"
        elif issue is not None:
            state = "invalid_will_be_rebuilt"
        elif method_stage_dir(run_dir, method).exists():
            state = "checkpoint_or_partial_stage"
        else:
            state = "not_started"
        rows.append({
            "seed": seed,
            "method": method,
            "state": state,
            "model": model_path.exists(),
            "utility": utility_path.exists(),
            "mia": mia_path.exists(),
            "runtime": runtime_path.exists(),
            "per_class": class_path.exists(),
            "issue": issue or "",
        })
    frame = pd.DataFrame(rows)
    display(frame)
    return frame

def run_one_method(
    cfg: ExperimentConfig,
    seed: int,
    method: str,
    force: bool = False,
) -> Dict[str, Path]:
    global ACTIVE_METHOD_DIR, ACTIVE_FIT_COUNTER, ACTIVE_METHOD_NAME

    if method not in set(cfg.enabled_methods) | {"head_reset"}:
        raise ValueError(f"Method is not configured: {method}")

    run_dir = seed_dir(cfg, seed)
    proxy_path = run_dir / "proxies" / "proxies.npz"
    if not proxy_path.exists():
        raise FileNotFoundError(
            f"Seed {seed} is not prepared. Run its Stage 3 cell first."
        )

    model_path = method_model_path(run_dir, method)
    utility_path = method_metric_path(run_dir, method, "utility")
    mia_path = method_metric_path(run_dir, method, "mia")
    runtime_path = method_metric_path(run_dir, method, "runtime")
    class_path = method_metric_path(run_dir, method, "per_class")
    metadata_path = run_dir / "metrics" / f"{method}_metadata.json"

    quality_issue = method_artifact_quality_issue(method, utility_path)
    if quality_issue is not None and not force:
        archive_dir = archive_invalid_method_artifacts(run_dir, method)
        print(f"[{method}][seed={seed}] {quality_issue}")
        print("Archived invalid artifacts to:", archive_dir)

    complete = all(
        path.exists()
        for path in (model_path, utility_path, mia_path, runtime_path, class_path)
    )
    if complete and not force:
        print(f"[{method}][seed={seed}] complete; skipping.")
        return {
            "model": model_path,
            "utility": utility_path,
            "mia": mia_path,
            "runtime": runtime_path,
            "per_class": class_path,
        }

    if force and method != "original":
        for path in (
            model_path,
            utility_path,
            mia_path,
            runtime_path,
            class_path,
            metadata_path,
        ):
            path.unlink(missing_ok=True)
        shutil.rmtree(method_stage_dir(run_dir, method), ignore_errors=True)

    reset_tensorflow(seed)
    bundle = load_dataset(cfg)
    forget_classes = cfg.forget_classes_by_dataset[cfg.dataset]
    split = make_unlearning_split(
        bundle,
        forget_classes,
        cfg.retrain_validation_fraction,
        seed,
    )
    original_metadata = load_original_metadata(cfg)
    output_is_logits = bool(original_metadata["output_is_logits"])
    original_path = method_model_path(run_dir, "original")

    proxies = np.empty((0,) + bundle.input_shape, dtype=np.float32)
    proxy_labels = np.empty((0,), dtype=np.int64)
    if method in {"scrubref_head", "scrubref_rep"}:
        with np.load(proxy_path) as cached:
            proxies = cached["images"].astype(np.float32, copy=True)
            proxy_labels = cached["labels"].astype(np.int64, copy=True)

    ACTIVE_METHOD_NAME = method
    ACTIVE_METHOD_DIR = method_stage_dir(run_dir, method)
    ACTIVE_FIT_COUNTER = 0
    ACTIVE_METHOD_DIR.mkdir(parents=True, exist_ok=True)
    resume_artifacts_before = any(ACTIVE_METHOD_DIR.rglob("*"))
    set_stage_status(
        cfg,
        seed,
        method,
        "running",
        resumed=resume_artifacts_before,
    )
    started = time.perf_counter()

    try:
        metadata: Dict[str, float] = {}
        reused_saved_model = float(model_path.exists() and not force)
        training_seconds = float("nan")
        training_peak_gpu_mb = 0.0

        if method == "original":
            if not model_path.exists():
                raise FileNotFoundError(f"Original model is missing: {model_path}")
            training_seconds = 0.0
            reused_saved_model = 1.0
        elif not model_path.exists() or force:
            original = load_saved_model(original_path)
            model, metadata = build_method(
                method,
                original,
                split,
                bundle,
                proxies,
                proxy_labels,
                forget_classes,
                output_is_logits,
                cfg,
                seed,
            )
            training_seconds = time.perf_counter() - started
            training_peak_gpu_mb = peak_gpu_memory_mb()
            if cfg.save_models:
                temporary_model = model_path.with_name(
                    model_path.stem + "_writing.keras"
                )
                temporary_model.unlink(missing_ok=True)
                model.save(temporary_model)
                temporary_model.replace(model_path)
            del model, original
            release_memory(clear_session=True)
        else:
            print(
                f"[{method}][seed={seed}] saved model found; "
                "skipping training and completing evaluation."
            )

        if not model_path.exists():
            raise FileNotFoundError(
                f"Method model was not saved successfully: {model_path}"
            )

        reset_tensorflow(seed)
        ACTIVE_METHOD_NAME = method
        model = load_saved_model(model_path)
        utility, class_accuracy = evaluate_utility(
            model,
            bundle,
            forget_classes,
            output_is_logits,
            cfg.eval_batch_size,
            seed,
        )
        mia = evaluate_membership_inference(
            model,
            split,
            bundle,
            output_is_logits,
            cfg.eval_batch_size,
            cfg.mia_attack_repeats,
            seed,
        )
        evaluation_peak_gpu_mb = peak_gpu_memory_mb()
        changed_weights = [
            np.asarray(weight).copy()
            for weight in model.get_weights()
        ]
        del model
        release_memory(clear_session=True)

        if method == "original":
            parameter_stats = {
                "changed_parameter_fraction": 0.0,
                "relative_l2_parameter_change": 0.0,
            }
        else:
            original_for_weights = load_saved_model(original_path)
            original_weights = [
                np.asarray(weight).copy()
                for weight in original_for_weights.get_weights()
            ]
            del original_for_weights
            release_memory(clear_session=True)
            parameter_stats = parameter_change_statistics_from_weights(
                original_weights,
                changed_weights,
            )
            del original_weights
        del changed_weights
        release_memory(clear_session=True)

        runtime_row = {
            "seed": seed,
            "method": method,
            "runtime_seconds": training_seconds,
            "peak_gpu_memory_mb": max(
                float(training_peak_gpu_mb),
                float(evaluation_peak_gpu_mb),
            ),
            "reused_saved_model": reused_saved_model,
            "resumed_from_checkpoint": float(resume_artifacts_before),
            "runtime_valid_for_comparison": float(
                not resume_artifacts_before and not reused_saved_model
            ),
            "output_is_logits": float(output_is_logits),
            **parameter_stats,
        }
        utility_row = {
            "seed": seed,
            "method": method,
            **utility,
            **metadata,
        }
        mia_row = {"seed": seed, "method": method, **mia}
        class_rows = [
            {
                "seed": seed,
                "method": method,
                "class_id": class_id,
                "class_name": bundle.class_names[class_id],
                "is_forgotten": int(class_id in forget_classes),
                "accuracy": float(accuracy),
            }
            for class_id, accuracy in enumerate(class_accuracy)
        ]

        atomic_write_dataframe(utility_path, pd.DataFrame([utility_row]))
        atomic_write_dataframe(mia_path, pd.DataFrame([mia_row]))
        atomic_write_dataframe(runtime_path, pd.DataFrame([runtime_row]))
        atomic_write_dataframe(class_path, pd.DataFrame(class_rows))
        atomic_write_json(
            metadata_path,
            {
                "method": method,
                "seed": seed,
                "training_seconds": training_seconds,
                "metadata": metadata,
                "completed_at": utc_now(),
                "v7_memory_safe_evaluation": True,
            },
        )
        set_stage_status(
            cfg,
            seed,
            method,
            "complete",
            utility=utility,
            mia=mia,
        )
        print(f"[{method}][seed={seed}] complete")
        print("Utility:", utility)
        print("MIA:", mia)
    except Exception as exc:
        set_stage_status(
            cfg,
            seed,
            method,
            "failed",
            error=repr(exc),
            traceback=traceback.format_exc(),
        )
        raise
    finally:
        ACTIVE_METHOD_NAME = None
        ACTIVE_METHOD_DIR = None
        ACTIVE_FIT_COUNTER = 0
        release_memory(clear_session=True)

    return {
        "model": model_path,
        "utility": utility_path,
        "mia": mia_path,
        "runtime": runtime_path,
        "per_class": class_path,
    }



def run_method_for_seeds(
    method: str,
    cfg: ExperimentConfig = CFG,
    seeds: Optional[Sequence[int]] = None,
    force: bool = False,
) -> None:
    selected = tuple(seeds or cfg.seeds)
    for seed in selected:
        print("\n" + "=" * 90)
        print(f"Method={method}, dataset={cfg.dataset}, seed={seed}")
        print("=" * 90)
        run_one_method(cfg, seed, method, force=force)

from src.utils import stage_status_path
def status_dashboard(
    cfg: ExperimentConfig = CFG,
    seeds: Optional[Sequence[int]] = None,
) -> pd.DataFrame:
    rows: List[Dict[str, object]] = []
    for seed in tuple(seeds or cfg.seeds):
        for stage in ("prepare",) + cfg.enabled_methods:
            path = stage_status_path(cfg, seed, stage)
            if path.exists():
                payload = json.loads(path.read_text(encoding="utf-8"))
                rows.append({
                    "seed": seed,
                    "stage": stage,
                    "state": payload.get("state", "unknown"),
                    "updated_at": payload.get("updated_at", ""),
                    "error": payload.get("error", ""),
                })
            else:
                rows.append({"seed": seed, "stage": stage, "state": "not_started", "updated_at": "", "error": ""})
    frame = pd.DataFrame(rows)
    display(frame)
    return frame


### Versioned Metric Migration and Evaluation Refresh

This section upgrades existing experiment artifacts without unnecessarily retraining saved models.

- `migrate_existing_mia_outputs_v7` updates legacy membership-inference CSV files by preserving the raw learned AUC and converting the reported value to an attack-direction-invariant AUC.
- `refresh_method_evaluation_v7` recomputes utility, privacy, and per-class metrics directly from a saved model while marking its runtime as unsuitable for direct wall-clock comparison.
- `continue_retrained_reference_v7` continues an existing retrained reference model for the additional v7 training phase, stores a completion marker, removes stale feature artifacts, and refreshes the resulting evaluation.
- `upgrade_seed_to_v7` applies the migration workflow to one seed and optionally continues the saved retraining reference.

This migration path preserves completed training artifacts while keeping metric schemas and evaluation outputs consistent across experiment versions.


In [ ]:
def migrate_existing_mia_outputs_v7(
    cfg: ExperimentConfig,
    seed: int,
) -> pd.DataFrame:
    """Upgrade existing MIA CSVs without rerunning models."""
    rows: List[Dict[str, object]] = []
    run_dir = seed_dir(cfg, seed)
    for method in cfg.core_methods:
        path = method_metric_path(run_dir, method, "mia")
        if not path.exists():
            rows.append({"seed": seed, "method": method, "state": "missing"})
            continue
        frame = pd.read_csv(path)
        if frame.empty or "mia_learned_auc" not in frame.columns:
            rows.append({"seed": seed, "method": method, "state": "invalid"})
            continue
        if "mia_learned_auc_raw" not in frame.columns:
            frame["mia_learned_auc_raw"] = frame["mia_learned_auc"].astype(float)
        raw = frame["mia_learned_auc_raw"].astype(float)
        frame["mia_learned_auc"] = np.maximum(raw, 1.0 - raw)
        frame["mia_schema_version"] = V7_AUDIT_VERSION
        atomic_write_dataframe(path, frame)
        rows.append({
            "seed": seed,
            "method": method,
            "state": "upgraded",
            "raw_auc": float(frame.iloc[0]["mia_learned_auc_raw"]),
            "corrected_auc": float(frame.iloc[0]["mia_learned_auc"]),
        })
    report = pd.DataFrame(rows)
    display(report)
    return report


def refresh_method_evaluation_v7(
    cfg: ExperimentConfig,
    seed: int,
    method: str,
    continuation_seconds: float = float("nan"),
) -> Dict[str, Path]:
    """Recompute utility, MIA, and per-class metrics from a saved model only."""
    run_dir = seed_dir(cfg, seed)
    model_path = method_model_path(run_dir, method)
    if not model_path.exists():
        raise FileNotFoundError(f"Saved model is missing: {model_path}")

    reset_tensorflow(seed)
    bundle = load_dataset(cfg)
    forget_classes = cfg.forget_classes_by_dataset[cfg.dataset]
    split = make_unlearning_split(
        bundle,
        forget_classes,
        cfg.retrain_validation_fraction,
        seed,
    )
    output_is_logits = bool(load_original_metadata(cfg)["output_is_logits"])
    model = load_saved_model(model_path)
    try:
        utility, class_accuracy = evaluate_utility(
            model,
            bundle,
            forget_classes,
            output_is_logits,
            cfg.eval_batch_size,
            seed,
        )
        mia = evaluate_membership_inference(
            model,
            split,
            bundle,
            output_is_logits,
            cfg.eval_batch_size,
            cfg.mia_attack_repeats,
            seed,
        )
    finally:
        del model
        release_memory(clear_session=True)

    utility_path = method_metric_path(run_dir, method, "utility")
    mia_path = method_metric_path(run_dir, method, "mia")
    class_path = method_metric_path(run_dir, method, "per_class")
    runtime_path = method_metric_path(run_dir, method, "runtime")

    atomic_write_dataframe(
        utility_path,
        pd.DataFrame([{"seed": seed, "method": method, **utility}]),
    )
    atomic_write_dataframe(
        mia_path,
        pd.DataFrame([{"seed": seed, "method": method, **mia, "mia_schema_version": V7_AUDIT_VERSION}]),
    )
    atomic_write_dataframe(
        class_path,
        pd.DataFrame([
            {
                "seed": seed,
                "method": method,
                "class_id": class_id,
                "class_name": bundle.class_names[class_id],
                "is_forgotten": int(class_id in forget_classes),
                "accuracy": float(accuracy),
            }
            for class_id, accuracy in enumerate(class_accuracy)
        ]),
    )

    if runtime_path.exists():
        runtime = pd.read_csv(runtime_path)
    else:
        runtime = pd.DataFrame([{"seed": seed, "method": method}])
    runtime["v7_continuation_seconds"] = continuation_seconds
    runtime["runtime_valid_for_comparison"] = 0.0
    runtime["runtime_note"] = "seed upgraded from saved checkpoint; utility/privacy valid, full wall-clock excluded"
    atomic_write_dataframe(runtime_path, runtime)

    set_stage_status(
        cfg,
        seed,
        method,
        "complete",
        utility=utility,
        mia=mia,
        v7_refreshed=True,
    )
    print(f"[{method}][seed={seed}] v7 evaluation refreshed")
    print("Utility:", utility)
    print("MIA:", mia)
    return {
        "model": model_path,
        "utility": utility_path,
        "mia": mia_path,
        "runtime": runtime_path,
        "per_class": class_path,
    }


def continue_retrained_reference_v7(
    cfg: ExperimentConfig,
    seed: int,
) -> Dict[str, Path]:
    """Continue an existing 35-epoch retraining model for the shared v7 phase two."""
    global ACTIVE_METHOD_DIR, ACTIVE_FIT_COUNTER, ACTIVE_METHOD_NAME

    run_dir = seed_dir(cfg, seed)
    model_path = method_model_path(run_dir, "retrained")
    if not model_path.exists():
        raise FileNotFoundError(
            "The phase-one retrained model is missing. Run the retrained Stage 4 cell instead."
        )

    phase_dir = run_dir / "stages" / "retrained_v7_phase2"
    marker = phase_dir / "complete.json"
    continuation_seconds = float("nan")

    if not marker.exists():
        reset_tensorflow(seed)
        bundle = load_dataset(cfg)
        split = make_unlearning_split(
            bundle,
            cfg.forget_classes_by_dataset[cfg.dataset],
            cfg.retrain_validation_fraction,
            seed,
        )
        output_is_logits = bool(load_original_metadata(cfg)["output_is_logits"])
        model = load_saved_model(model_path)

        ACTIVE_METHOD_NAME = "retrained_v7_phase2"
        ACTIVE_METHOD_DIR = phase_dir
        ACTIVE_FIT_COUNTER = 0
        phase_dir.mkdir(parents=True, exist_ok=True)
        started = time.perf_counter()
        set_stage_status(cfg, seed, "retrained_v7_phase2", "running")
        try:
            model = train_supervised(
                model,
                split.retain_train_items,
                split.retain_train_labels,
                bundle,
                output_is_logits,
                V7_RETRAIN_EXTRA_EPOCHS,
                cfg.batch_size,
                V7_RETRAIN_PHASE2_LR,
                seed + 1000,
                validation=(split.retain_val_items, split.retain_val_labels),
                train_all=True,
            )
            continuation_seconds = time.perf_counter() - started
            temporary = model_path.with_name(model_path.stem + "_v7_writing.keras")
            temporary.unlink(missing_ok=True)
            model.save(temporary)
            temporary.replace(model_path)
            atomic_write_json(marker, {
                "seed": seed,
                "extra_epochs": V7_RETRAIN_EXTRA_EPOCHS,
                "learning_rate": V7_RETRAIN_PHASE2_LR,
                "continuation_seconds": continuation_seconds,
                "completed_at": utc_now(),
            })
            set_stage_status(
                cfg,
                seed,
                "retrained_v7_phase2",
                "complete",
                continuation_seconds=continuation_seconds,
            )
        except Exception as exc:
            set_stage_status(
                cfg,
                seed,
                "retrained_v7_phase2",
                "failed",
                error=repr(exc),
                traceback=traceback.format_exc(),
            )
            raise
        finally:
            del model
            ACTIVE_METHOD_NAME = None
            ACTIVE_METHOD_DIR = None
            ACTIVE_FIT_COUNTER = 0
            release_memory(clear_session=True)
    else:
        payload = json.loads(marker.read_text(encoding="utf-8"))
        continuation_seconds = float(payload.get("continuation_seconds", float("nan")))
        print(f"[retrained][seed={seed}] v7 phase two already complete; skipping training.")

    for stale in (
        run_dir / "features" / "retrained.npz",
        run_dir / "features" / "retrained_probs.npz",
        run_dir / "representation_metrics.csv",
    ):
        stale.unlink(missing_ok=True)

    return refresh_method_evaluation_v7(
        cfg,
        seed,
        "retrained",
        continuation_seconds=continuation_seconds,
    )


def upgrade_seed_to_v7(
    cfg: ExperimentConfig,
    seed: int,
    continue_existing_retrained: bool = False,
) -> None:
    print(f"Applying v7 migration to seed {seed}")
    migrate_existing_mia_outputs_v7(cfg, seed)
    if continue_existing_retrained:
        continue_retrained_reference_v7(cfg, seed)
    else:
        print("No existing retraining continuation requested for this seed.")
    print("v7 seed upgrade complete.")


## Stage 2: Preflight & Required Checkpoint Validation

This stage performs initial validation checks before running core experiments:
- **Directory & Configuration Fingerprint:** Resolves experiment paths and prints the unique configuration hash for reproducibility.
- **Original Model Verification:** Checks whether the pre-trained original model checkpoint exists for the selected dataset.
- **Conditional Training:** If `CFG.require_original_checkpoint` is enabled and the checkpoint is missing, it triggers an on-the-fly training routine; otherwise, it logs the status and proceeds safely.



In [ ]:
from src.utils import experiment_dir, config_fingerprint, reset_tensorflow, seed_dir

print("Experiment directory:", experiment_dir(CFG))
print("Configuration fingerprint:", config_fingerprint(CFG))

checkpoint = Path(CFG.checkpoint_paths[CFG.dataset])
checkpoint_exists = checkpoint.exists()

print(f"Required original checkpoint: {checkpoint}")
print(f"Checkpoint exists: {checkpoint_exists}")

if not checkpoint_exists:
    if CFG.require_original_checkpoint:
        print("Checkpoint not found. Training the original model in this stage...")

        # Replace this with the actual function used to train the original model.
        reset_tensorflow(42)
        bundle = load_dataset(CFG)
        print(f"Dataset Loaded: {bundle.name} | Input Shape: {bundle.input_shape}")
        run_dir = seed_dir(CFG, 42)
        run_dir.mkdir(parents=True, exist_ok=True)
        model, output_is_logits, source_path = load_or_train_original(CFG, bundle, 42, run_dir)

        if checkpoint.exists():
            print(f"Original model trained and saved: {checkpoint}")
        else:
            raise FileNotFoundError(
                "Training finished, but the expected checkpoint was not found: "
                f"{checkpoint}"
            )
    else:
        print(
            "Original-model checkpoint is missing. "
            "This stage will continue; the model is expected to be trained later."
        )
else:
    print("Original-model checkpoint found. Continuing.")


Dataset Preflight Check (RAM & Type Safety)

In [ ]:
PREFLIGHT_BUNDLE = load_dataset(CFG)
print("Train:", PREFLIGHT_BUNDLE.train_items.shape, PREFLIGHT_BUNDLE.train_items.dtype)
print("Test:", PREFLIGHT_BUNDLE.test_items.shape, PREFLIGHT_BUNDLE.test_items.dtype)
print(
    "Dataset image RAM:",
    f"{array_memory_mb(PREFLIGHT_BUNDLE.train_items, PREFLIGHT_BUNDLE.test_items):.2f} MB",
)
if not PREFLIGHT_BUNDLE.path_mode and PREFLIGHT_BUNDLE.train_items.dtype != np.uint8:
    raise RuntimeError("RAM preflight failed: CIFAR images are not uint8.")
del PREFLIGHT_BUNDLE
release_memory(clear_session=False)
print("Preflight passed.")

## Stage 3 — prepare seed 42 and generate resumable class proxies

Run the seed-42 preparation cell. Proxy generation is checkpointed in Google Drive and resumes after disconnects.

In [ ]:
prepare_seed(CFG, 42)


In [ ]:
status_dashboard(CFG, seeds=(42,))

## Stage 4 — train and evaluate one method per cell


Rerun only the interrupted method cell after reconnecting. Completed seed-method pairs are skipped.


### GPU OOM recovery before Stage 4

After a `ResourceExhaustedError`, restart the Colab runtime to clear TensorFlow's GPU allocator.

Then:

1. Rerun Stages 0–2.
2. Run the v6 resume report.
3. Rerun only cells marked `model_saved_evaluation_pending`, `checkpoint_or_partial_stage`, `invalid_will_be_rebuilt`, or `not_started`.

Do not regenerate completed proxies. Do not force-rerun completed methods.


In [ ]:
# Verify that the existing seed-42 proxy and shared model will be reused.
seed42_dir = seed_dir(CFG, 42)

print("Experiment directory:", experiment_dir(CFG))
print("Proxy exists:", (seed42_dir / "proxies.npz").exists())
print("Original exists:", method_model_path(seed42_dir, "original").exists())

# Dynamically use CFG.dataset to ensure correct dataset name labeling in logs
print(
    f"Effective {CFG.dataset} training batch:",
    effective_runtime_batch_size(load_dataset(CFG), CFG.batch_size, True),
)
print(
    f"Effective {CFG.dataset} evaluation batch:",
    effective_runtime_batch_size(load_dataset(CFG), CFG.eval_batch_size, False),
)

### Original model evaluation


**original — seed 42**


In [ ]:
run_one_method(CFG, 42, "original")


In [ ]:
status_dashboard(CFG, seeds=(42,))

### Retain-only retraining reference; longest single stage, with Keras backup/restore


**retrained — seed 42**


In [ ]:
run_one_method(CFG, 42, "retrained")


In [ ]:
status_dashboard(CFG, seeds=(42,))

### Trivial classifier-output suppression control


**logit_clamp — seed 42**


In [ ]:
run_one_method(CFG, 42, "logit_clamp")

In [ ]:
status_dashboard(CFG, seeds=(42,))

### Retained-data fine-tuning control


**retain_finetune — seed 42**


In [ ]:
run_one_method(CFG, 42, "retain_finetune")

In [ ]:
status_dashboard(CFG, seeds=(42,))

### Corrected head-only ScrubRef


**scrubref_head — seed 42**


In [ ]:
run_one_method(CFG, 42, "scrubref_head")

In [ ]:
status_dashboard(CFG, seeds=(42,))

### Representation-aware ScrubRef


**scrubref_rep — seed 42**


In [ ]:
run_one_method(CFG, 42, "scrubref_rep")

In [ ]:
status_dashboard(CFG, seeds=(42,))

### UNSIR-style impair-and-repair baseline


**unsir — seed 42**


In [ ]:
run_one_method(CFG, 42, "unsir")

In [ ]:
status_dashboard(CFG, seeds=(42,))

### Bad Teacher baseline


**bad_teacher — seed 42**


In [ ]:
run_one_method(CFG, 42, "bad_teacher")

In [ ]:
status_dashboard(CFG, seeds=(42,))

### SCRUB-style baseline


**scrub — seed 42**


In [ ]:
run_one_method(CFG, 42, "scrub")

In [ ]:
status_dashboard(CFG, seeds=(42,))

### Selective synaptic dampening baseline


**ssd — seed 42**


In [ ]:
run_one_method(CFG, 42, "ssd")

In [ ]:
status_dashboard(CFG, seeds=(42,))

### Optional head-reset control


In [ ]:
RUN_OPTIONAL_HEAD_RESET = False
if RUN_OPTIONAL_HEAD_RESET:
    run_one_method(CFG, 42, "head_reset")
else:
    print("Optional head-reset control skipped.")

## Stage 5 — representation extraction for seed 42

Run both extraction groups. Feature arrays are saved to Google Drive and reused after reconnects.

In [ ]:
def representation_index_path(cfg: ExperimentConfig, seed: int) -> Path:
    return seed_dir(cfg, seed) / "features" / f"audit_indices_v{V7_AUDIT_VERSION}.npz"


def ensure_representation_indices(cfg: ExperimentConfig, bundle: DataBundle, seed: int) -> Dict[str, np.ndarray]:
    path = representation_index_path(cfg, seed)
    if path.exists():
        with np.load(path) as cached:
            version = int(cached["audit_version"][0]) if "audit_version" in cached.files else 0
            if version == V7_AUDIT_VERSION:
                return {
                    name: cached[name].astype(np.int64, copy=False)
                    for name in ("train_idx", "test_idx", "cka_idx")
                }

    forget_classes = np.asarray(cfg.forget_classes_by_dataset[cfg.dataset], dtype=np.int64)
    base_train = stratified_indices(bundle.train_labels, cfg.probe_train_limit, seed)
    base_test = stratified_indices(bundle.test_labels, cfg.probe_test_limit, seed + 1)
    base_cka = stratified_indices(bundle.test_labels, cfg.cka_sample_limit, seed + 2)

    forgotten_train = np.flatnonzero(np.isin(bundle.train_labels, forget_classes))
    forgotten_test = np.flatnonzero(np.isin(bundle.test_labels, forget_classes))

    indices = {
        "train_idx": np.unique(np.concatenate([base_train, forgotten_train])).astype(np.int64),
        "test_idx": np.unique(np.concatenate([base_test, forgotten_test])).astype(np.int64),
        "cka_idx": np.unique(np.concatenate([base_cka, forgotten_test])).astype(np.int64),
    }
    np.savez_compressed(
        path,
        **indices,
        audit_version=np.asarray([V7_AUDIT_VERSION], dtype=np.int64),
        forgotten_train_count=np.asarray([len(forgotten_train)], dtype=np.int64),
        forgotten_test_count=np.asarray([len(forgotten_test)], dtype=np.int64),
    )
    print({
        "audit_version": V7_AUDIT_VERSION,
        "train_probe_count": len(indices["train_idx"]),
        "test_probe_count": len(indices["test_idx"]),
        "cka_count": len(indices["cka_idx"]),
        "forgotten_train_included": len(forgotten_train),
        "forgotten_test_included": len(forgotten_test),
    })
    return indices


def representation_artifacts_are_v7(feature_path: Path, probability_path: Path) -> bool:
    if not feature_path.exists() or not probability_path.exists():
        return False
    try:
        with np.load(feature_path) as data:
            if "audit_version" not in data.files:
                return False
            return int(data["audit_version"][0]) == V7_AUDIT_VERSION
    except Exception:
        return False

def extract_representation_for_method(
    cfg: ExperimentConfig,
    seed: int,
    method: str,
    force: bool = False,
) -> Path:
    run_dir = seed_dir(cfg, seed)
    model_path = method_model_path(run_dir, method)
    if not model_path.exists():
        raise FileNotFoundError(f"Method model is missing: {model_path}")
    feature_path = run_dir / "features" / f"{method}.npz"
    probability_path = run_dir / "features" / f"{method}_probs.npz"
    if representation_artifacts_are_v7(feature_path, probability_path) and not force:
        print(f"[features][{method}][seed={seed}] v7 cached")
        return feature_path
    if feature_path.exists() or probability_path.exists():
        print(f"[features][{method}][seed={seed}] stale pre-v7 artifacts detected; rebuilding")

    reset_tensorflow(seed)
    bundle = load_dataset(cfg)
    indices = ensure_representation_indices(cfg, bundle, seed)
    original_metadata = load_original_metadata(cfg)
    output_is_logits = bool(original_metadata["output_is_logits"])
    model = load_saved_model(model_path)
    set_stage_status(cfg, seed, f"features_{method}", "running")
    try:
        train_features, train_labels = extract_features(
            model,
            bundle.train_items[indices["train_idx"]],
            bundle.train_labels[indices["train_idx"]],
            bundle,
            cfg.eval_batch_size,
            seed,
        )
        test_features, test_labels = extract_features(
            model,
            bundle.test_items[indices["test_idx"]],
            bundle.test_labels[indices["test_idx"]],
            bundle,
            cfg.eval_batch_size,
            seed,
        )
        cka_features, cka_labels = extract_features(
            model,
            bundle.test_items[indices["cka_idx"]],
            bundle.test_labels[indices["cka_idx"]],
            bundle,
            cfg.eval_batch_size,
            seed,
        )
        probabilities, probability_labels = predict_probabilities(
            model,
            bundle.test_items[indices["cka_idx"]],
            bundle.test_labels[indices["cka_idx"]],
            bundle,
            output_is_logits,
            cfg.eval_batch_size,
            seed,
        )
        np.savez_compressed(
            feature_path,
            train_features=train_features.astype(np.float32, copy=False),
            train_labels=train_labels,
            test_features=test_features.astype(np.float32, copy=False),
            test_labels=test_labels,
            cka_features=cka_features.astype(np.float32, copy=False),
            cka_labels=cka_labels,
            audit_version=np.asarray([V7_AUDIT_VERSION], dtype=np.int64),
            forgotten_train_count=np.asarray([np.sum(np.isin(train_labels, cfg.forget_classes_by_dataset[cfg.dataset]))], dtype=np.int64),
            forgotten_test_count=np.asarray([np.sum(np.isin(test_labels, cfg.forget_classes_by_dataset[cfg.dataset]))], dtype=np.int64),
        )
        np.savez_compressed(
            probability_path,
            probabilities=probabilities.astype(np.float32, copy=False),
            labels=probability_labels,
            audit_version=np.asarray([V7_AUDIT_VERSION], dtype=np.int64),
        )
        set_stage_status(cfg, seed, f"features_{method}", "complete")
        print(f"[features][{method}][seed={seed}] complete")
    except Exception as exc:
        set_stage_status(cfg, seed, f"features_{method}", "failed", error=repr(exc), traceback=traceback.format_exc())
        raise
    finally:
        release_memory(clear_session=True)
    return feature_path


def extract_representation_group(
    methods: Sequence[str],
    cfg: ExperimentConfig = CFG,
    seeds: Optional[Sequence[int]] = None,
) -> None:
    for seed in tuple(seeds or cfg.seeds):
        for method in methods:
            extract_representation_for_method(cfg, seed, method)


### Core representation group


In [ ]:
CORE_REPRESENTATION_METHODS = (
    "original",
    "retrained",
    "logit_clamp",
    "retain_finetune",
    "scrubref_head",
    "scrubref_rep",
)


**Core representation extraction — seed 42**

In [ ]:
extract_representation_group(CORE_REPRESENTATION_METHODS, seeds=(42,))

### Baseline representation group


In [ ]:
BASELINE_REPRESENTATION_METHODS = (
    "unsir",
    "bad_teacher",
    "scrub",
    "ssd",
)

**Baseline representation extraction — seed 42**

In [ ]:
extract_representation_group(BASELINE_REPRESENTATION_METHODS, seeds=(42,))

## Stage 6 — compute representation metrics for seed 42

In [ ]:
def compute_representation_metrics_for_seed(
    cfg: ExperimentConfig,
    seed: int,
    force: bool = False,
) -> Path:
    run_dir = seed_dir(cfg, seed)
    output_path = run_dir / "representation_metrics.csv"
    required_methods = v7_representation_methods(cfg)
    if output_path.exists() and not force:
        cached = pd.read_csv(output_path)
        cached_methods = set(cached.get("method", pd.Series(dtype=str)).astype(str))
        cache_is_v7 = (
            "audit_version" in cached.columns
            and not cached.empty
            and bool(np.all(cached["audit_version"].astype(int) == V7_AUDIT_VERSION))
            and set(required_methods).issubset(cached_methods)
        )
        if cache_is_v7:
            print("Representation metrics v7 cached:", output_path)
            return output_path
        print("Stale representation metrics detected; rebuilding:", output_path)

    feature_paths = {
        method: run_dir / "features" / f"{method}.npz"
        for method in required_methods
        if (run_dir / "features" / f"{method}.npz").exists()
    }
    probability_paths = {
        method: run_dir / "features" / f"{method}_probs.npz"
        for method in required_methods
        if (run_dir / "features" / f"{method}_probs.npz").exists()
    }
    if "original" not in feature_paths or "retrained" not in feature_paths:
        raise RuntimeError("Original and retrained feature files are required.")

    with np.load(feature_paths["original"]) as data:
        original_features = data["cka_features"].astype(np.float32, copy=True)
    with np.load(feature_paths["retrained"]) as data:
        retrained_features = data["cka_features"].astype(np.float32, copy=True)
    with np.load(probability_paths["retrained"]) as data:
        retrained_probs = data["probabilities"].astype(np.float32, copy=True)
        labels = data["labels"].astype(np.int64, copy=True)

    forget_classes = cfg.forget_classes_by_dataset[cfg.dataset]
    forget_mask = np.isin(labels, np.asarray(forget_classes))
    retain_mask = ~forget_mask
    rows: List[Dict[str, object]] = []

    for method, feature_path in feature_paths.items():
        print(f"Representation metrics: method={method}, seed={seed}")
        with np.load(feature_path) as data:
            train_features = data["train_features"].astype(np.float32, copy=True)
            train_labels = data["train_labels"].astype(np.int64, copy=True)
            test_features = data["test_features"].astype(np.float32, copy=True)
            test_labels = data["test_labels"].astype(np.int64, copy=True)
            method_features = data["cka_features"].astype(np.float32, copy=True)

        probe = representation_probe_metrics(
            train_features,
            train_labels,
            test_features,
            test_labels,
            forget_classes,
        )
        forgotten_test_count = int(np.sum(np.isin(test_labels, np.asarray(forget_classes))))
        row: Dict[str, object] = {
            "seed": seed,
            "method": method,
            "audit_version": V7_AUDIT_VERSION,
            "probe_test_forgotten_count": forgotten_test_count,
            "cka_to_original": linear_cka(method_features, original_features),
            "cka_to_retrained": linear_cka(method_features, retrained_features),
            **probe,
        }
        if method in probability_paths:
            with np.load(probability_paths[method]) as data:
                method_probs = data["probabilities"].astype(np.float32, copy=True)
            row.update({
                "js_to_retrained_all": jensen_shannon_divergence(method_probs, retrained_probs),
                "js_to_retrained_retain": jensen_shannon_divergence(
                    method_probs[retain_mask], retrained_probs[retain_mask]
                ),
                "js_to_retrained_forget": jensen_shannon_divergence(
                    method_probs[forget_mask], retrained_probs[forget_mask]
                ),
                "prediction_disagreement_to_retrained": float(
                    np.mean(method_probs.argmax(axis=1) != retrained_probs.argmax(axis=1))
                ),
            })
            del method_probs
        rows.append(row)
        del train_features, train_labels, test_features, test_labels, method_features, probe
        release_memory(clear_session=False)

    atomic_write_dataframe(output_path, pd.DataFrame(rows))
    set_stage_status(cfg, seed, "representation_metrics", "complete", method_count=len(rows))
    release_memory(clear_session=True)
    print("Saved:", output_path)
    return output_path


In [ ]:
compute_representation_metrics_for_seed(CFG, 42)


## Stage 7 — final single-seed consolidation for ImageWoof

This stage validates every method for seed 42, merges utility, calibration, membership-inference, runtime, and representation evidence, and creates publication tables and figures. It makes no multi-seed statistical claim.

In [ ]:
FINAL_SEED = 42

METHOD_ROLE = {
    "original": "Untouched reference",
    "retrained": "Retain-only retraining reference",
    "logit_clamp": "Output-only control",
    "retain_finetune": "Retained-data fine-tuning control",
    "scrubref_head": "Proposed method: head-only",
    "scrubref_rep": "Proposed method: representation-aware",
    "unsir": "UNSIR baseline",
    "bad_teacher": "Bad Teacher baseline",
    "scrub": "SCRUB baseline",
    "ssd": "SSD baseline",
}

METHOD_ORDER = list(CFG.enabled_methods)


def _read_single_row(path: Path) -> dict:
    if not path.exists():
        raise FileNotFoundError(path)
    frame = pd.read_csv(path)
    if frame.empty:
        raise RuntimeError(f"Empty result file: {path}")
    return frame.iloc[0].to_dict()


def build_single_seed_publication_package(
    cfg: ExperimentConfig,
    seed: int = FINAL_SEED,
) -> dict:
    run_dir = seed_dir(cfg, seed)
    aggregate_dir = experiment_dir(cfg) / "aggregate_seed42"
    figure_dir = aggregate_dir / "figures"
    aggregate_dir.mkdir(parents=True, exist_ok=True)
    figure_dir.mkdir(parents=True, exist_ok=True)

    representation_path = run_dir / "representation_metrics.csv"
    if not representation_path.exists():
        raise FileNotFoundError(
            f"Missing {representation_path}. Finish Stage 6 before Stage 7."
        )
    representation = pd.read_csv(representation_path)
    if representation.empty:
        raise RuntimeError("representation_metrics.csv is empty")
    representation = representation.set_index("method", drop=False)

    rows = []
    completeness_rows = []
    for method in METHOD_ORDER:
        utility_path = method_metric_path(run_dir, method, "utility")
        mia_path = method_metric_path(run_dir, method, "mia")
        runtime_path = method_metric_path(run_dir, method, "runtime")
        per_class_path = method_metric_path(run_dir, method, "per_class")
        model_path = method_model_path(run_dir, method)

        complete = all(
            path.exists()
            for path in (utility_path, mia_path, runtime_path, per_class_path, model_path)
        ) and method in representation.index
        completeness_rows.append({
            "seed": seed,
            "method": method,
            "model": model_path.exists(),
            "utility": utility_path.exists(),
            "mia": mia_path.exists(),
            "runtime": runtime_path.exists(),
            "per_class": per_class_path.exists(),
            "representation": method in representation.index,
            "complete": complete,
        })
        if not complete:
            continue

        utility = _read_single_row(utility_path)
        mia = _read_single_row(mia_path)
        runtime = _read_single_row(runtime_path)
        rep = representation.loc[method].to_dict()

        row = {
            "seed": seed,
            "method": method,
            "method_role": METHOD_ROLE.get(method, method),
            "is_scrubref": method.startswith("scrubref"),
        }
        for source in (utility, mia, runtime, rep):
            for key, value in source.items():
                if key not in {"seed", "method"}:
                    row[key] = value
        rows.append(row)

    completeness = pd.DataFrame(completeness_rows)
    atomic_write_dataframe(aggregate_dir / "publication_completeness.csv", completeness)
    incomplete = completeness[~completeness["complete"]]
    if not incomplete.empty:
        display(incomplete)
        raise RuntimeError("Incomplete method artifacts remain. Run the missing stages first.")

    summary = pd.DataFrame(rows)
    order_map = {method: index for index, method in enumerate(METHOD_ORDER)}
    summary["method_order"] = summary["method"].map(order_map)
    summary = summary.sort_values("method_order").drop(columns="method_order").reset_index(drop=True)

    original_retained = float(summary.loc[summary["method"] == "original", "retained_accuracy"].iloc[0])
    retrained_retained = float(summary.loc[summary["method"] == "retrained", "retained_accuracy"].iloc[0])
    retrained_probe_forget = float(summary.loc[summary["method"] == "retrained", "linear_probe_forgotten_accuracy"].iloc[0])
    summary["retained_drop_vs_original_pp"] = 100.0 * (original_retained - summary["retained_accuracy"])
    summary["retained_gap_vs_retrained_pp"] = 100.0 * (summary["retained_accuracy"] - retrained_retained)
    summary["forgotten_accuracy_pct"] = 100.0 * summary["forgotten_accuracy"]
    summary["retained_accuracy_pct"] = 100.0 * summary["retained_accuracy"]
    summary["learned_mia_excess_over_random"] = summary["mia_learned_auc"] - 0.5
    summary["output_forgetting_pass"] = summary["forgotten_accuracy"] <= 0.05
    summary["empirical_mia_pass"] = summary["mia_learned_auc"] <= 0.60
    summary["utility_vs_retrained_pass"] = summary["retained_gap_vs_retrained_pp"].abs() <= 3.0
    summary["representation_changed"] = summary["cka_to_original"] < 0.95
    summary["closer_to_retrained_than_original"] = summary["cka_to_retrained"] > summary["cka_to_original"]
    summary["probe_gap_vs_retrained_pp"] = 100.0 * (summary["linear_probe_forgotten_accuracy"] - retrained_probe_forget)
    summary["probe_reference_pass"] = summary["probe_gap_vs_retrained_pp"].abs() <= 10.0
    summary["representation_reference_pass"] = (
        summary["representation_changed"]
        & (summary["closer_to_retrained_than_original"] | summary["probe_reference_pass"])
    )
    summary["runtime_reportable"] = summary.get(
        "runtime_valid_for_comparison",
        pd.Series(False, index=summary.index),
    ).fillna(0).astype(bool)

    atomic_write_dataframe(aggregate_dir / "single_seed_method_summary.csv", summary)

    main_columns = [
        "method",
        "method_role",
        "retained_accuracy_pct",
        "forgotten_accuracy_pct",
        "mia_learned_auc",
        "nll",
        "ece_15",
        "cka_to_original",
        "cka_to_retrained",
        "linear_probe_forgotten_accuracy",
        "runtime_seconds",
        "runtime_reportable",
    ]
    main_table = summary[main_columns].copy()
    atomic_write_dataframe(aggregate_dir / "paper_table_main.csv", main_table)

    representation_columns = [
        "method",
        "cka_to_original",
        "cka_to_retrained",
        "linear_probe_overall_accuracy",
        "linear_probe_forgotten_accuracy",
        "linear_probe_retained_accuracy",
        "knn_overall_accuracy",
        "knn_forgotten_accuracy",
        "knn_retained_accuracy",
        "js_to_retrained_all",
        "js_to_retrained_retain",
        "js_to_retrained_forget",
        "prediction_disagreement_to_retrained",
        "probe_test_forgotten_count",
    ]
    atomic_write_dataframe(
        aggregate_dir / "paper_table_representation.csv",
        summary[representation_columns],
    )

    runtime_table = summary[[
        "method",
        "runtime_seconds",
        "runtime_valid_for_comparison",
        "reused_saved_model",
        "resumed_from_checkpoint",
        "peak_gpu_memory_mb",
        "changed_parameter_fraction",
        "relative_l2_parameter_change",
    ]].copy()
    atomic_write_dataframe(aggregate_dir / "paper_table_runtime.csv", runtime_table)

    design_methods = [
        "original",
        "retrained",
        "logit_clamp",
        "retain_finetune",
        "scrubref_head",
        "scrubref_rep",
    ]
    design_comparison = summary[summary["method"].isin(design_methods)][[
        "method",
        "method_role",
        "retained_accuracy_pct",
        "forgotten_accuracy_pct",
        "mia_learned_auc",
        "cka_to_original",
        "cka_to_retrained",
        "linear_probe_forgotten_accuracy",
        "retained_gap_vs_retrained_pp",
    ]].copy()
    atomic_write_dataframe(
        aggregate_dir / "scrubref_design_comparison.csv",
        design_comparison,
    )

    readiness = summary[summary["method"].isin(["scrubref_head", "scrubref_rep"])][[
        "method",
        "retained_drop_vs_original_pp",
        "retained_gap_vs_retrained_pp",
        "forgotten_accuracy_pct",
        "mia_learned_auc",
        "cka_to_original",
        "cka_to_retrained",
        "linear_probe_forgotten_accuracy",
        "probe_gap_vs_retrained_pp",
        "closer_to_retrained_than_original",
        "output_forgetting_pass",
        "empirical_mia_pass",
        "utility_vs_retrained_pass",
        "representation_changed",
        "probe_reference_pass",
        "representation_reference_pass",
    ]].copy()
    readiness["all_primary_checks_pass"] = readiness[[
        "output_forgetting_pass",
        "empirical_mia_pass",
        "utility_vs_retrained_pass",
        "representation_reference_pass",
    ]].all(axis=1)
    atomic_write_dataframe(aggregate_dir / "reviewer_readiness_checks.csv", readiness)

    evidence = pd.DataFrame([
        {
            "reviewer_concern": "Full retraining reference",
            "evidence": "retrained utility, MIA, calibration, and representation rows",
            "artifact": "paper_table_main.csv; paper_table_representation.csv",
            "status": "complete",
        },
        {
            "reviewer_concern": "Comparison with established baselines",
            "evidence": "UNSIR, Bad Teacher, SCRUB, and SSD under one evaluation protocol",
            "artifact": "single_seed_method_summary.csv",
            "status": "complete for seed 42",
        },
        {
            "reviewer_concern": "Output suppression is not representation unlearning",
            "evidence": "logit-clamp and head-only CKA versus representation-aware ScrubRef",
            "artifact": "scrubref_design_comparison.csv",
            "status": "complete",
        },
        {
            "reviewer_concern": "Membership-inference evaluation",
            "evidence": "direction-corrected learned attack plus confidence/loss/entropy attacks",
            "artifact": "paper_table_main.csv and per-method *_mia.csv files",
            "status": "complete",
        },
        {
            "reviewer_concern": "Calibration and predictive quality",
            "evidence": "NLL, Brier score, ECE, top-5 accuracy, macro-F1, balanced accuracy",
            "artifact": "paper_table_main.csv and per-method *_utility.csv files",
            "status": "complete",
        },
        {
            "reviewer_concern": "Representation analysis",
            "evidence": "CKA, linear probe, kNN, JS divergence, prediction disagreement",
            "artifact": "paper_table_representation.csv",
            "status": "complete",
        },
        {
            "reviewer_concern": "Statistical stability",
            "evidence": "Fixed seed 42 only",
            "artifact": "config.json",
            "status": "not a multi-seed claim",
        },
    ])
    atomic_write_dataframe(aggregate_dir / "reviewer_evidence_map.csv", evidence)

    # Accuracy figure.
    positions = np.arange(len(summary))
    width = 0.36
    fig, ax = plt.subplots(figsize=(max(11, len(summary) * 1.15), 5.8))
    ax.bar(positions - width / 2, summary["retained_accuracy"], width, label="Retained accuracy")
    ax.bar(positions + width / 2, summary["forgotten_accuracy"], width, label="Forgotten accuracy")
    ax.set_xticks(positions, summary["method"], rotation=35, ha="right")
    ax.set_ylim(0.0, 1.0)
    ax.set_ylabel("Accuracy")
    ax.legend()
    fig.tight_layout()
    fig.savefig(figure_dir / "seed42_accuracy.png", dpi=240)
    plt.close(fig)

    # Utility/privacy trade-off.
    fig, ax = plt.subplots(figsize=(8.2, 6.2))
    for _, row in summary.iterrows():
        ax.scatter(row["mia_learned_auc"], row["retained_accuracy"], s=70)
        ax.annotate(row["method"], (row["mia_learned_auc"], row["retained_accuracy"]), xytext=(5, 4), textcoords="offset points")
    ax.axvline(0.5, linestyle="--", linewidth=1.0)
    ax.set_xlim(0.48, max(0.80, float(summary["mia_learned_auc"].max()) + 0.03))
    ax.set_ylim(0.0, 0.9)
    ax.set_xlabel("Direction-corrected learned MIA ROC-AUC (lower is better)")
    ax.set_ylabel("Retained accuracy (higher is better)")
    fig.tight_layout()
    fig.savefig(figure_dir / "seed42_utility_privacy_tradeoff.png", dpi=240)
    plt.close(fig)

    # Representation similarity figure.
    fig, ax = plt.subplots(figsize=(max(11, len(summary) * 1.15), 5.8))
    ax.bar(positions - width / 2, summary["cka_to_original"], width, label="CKA to original")
    ax.bar(positions + width / 2, summary["cka_to_retrained"], width, label="CKA to retrained")
    ax.set_xticks(positions, summary["method"], rotation=35, ha="right")
    ax.set_ylim(0.0, 1.0)
    ax.set_ylabel("Linear CKA")
    ax.legend()
    fig.tight_layout()
    fig.savefig(figure_dir / "seed42_representation_cka.png", dpi=240)
    plt.close(fig)

    scrubref_rep = summary.loc[summary["method"] == "scrubref_rep"].iloc[0]
    scrubref_head = summary.loc[summary["method"] == "scrubref_head"].iloc[0]
    retrained = summary.loc[summary["method"] == "retrained"].iloc[0]
    original = summary.loc[summary["method"] == "original"].iloc[0]

    narrative = f"""# ScrubRef single-seed results handoff

Dataset: {cfg.dataset}
Seed: {seed}
Experiment directory: {experiment_dir(cfg)}

## Primary result

The representation-aware ScrubRef variant achieved retained accuracy of {100.0 * scrubref_rep['retained_accuracy']:.2f}%, forgotten accuracy of {100.0 * scrubref_rep['forgotten_accuracy']:.2f}%, and a direction-corrected learned membership-inference ROC-AUC of {scrubref_rep['mia_learned_auc']:.3f}. Its retained accuracy was {100.0 * (scrubref_rep['retained_accuracy'] - retrained['retained_accuracy']):+.2f} percentage points relative to retain-only retraining.

## Representation evidence

ScrubRef-Rep had CKA {scrubref_rep['cka_to_original']:.3f} to the original model and {scrubref_rep['cka_to_retrained']:.3f} to the retrained reference. The forgotten-class linear-probe accuracy was {100.0 * scrubref_rep['linear_probe_forgotten_accuracy']:.2f}%. The output-only logit-clamp and head-only ScrubRef controls should not be described as representation erasure when their CKA to the original model remains near one.

## Important limitations

These are fixed-seed results, not a multi-seed statistical claim. The untouched model retained accuracy was {100.0 * original['retained_accuracy']:.2f}%, while the retain-only retraining reference reached {100.0 * retrained['retained_accuracy']:.2f}%. Runtime comparisons must exclude rows where runtime_valid_for_comparison is false. UNSIR and any other collapsed baseline should be reported transparently as unstable under the evaluated configuration rather than treated as competitive.

## Recommended paper wording

Describe privacy as empirical resistance under the evaluated attacks, not as a formal privacy guarantee. Describe ScrubRef-Rep as changing internal representations and reducing forgotten-class evidence, but do not claim that its representation matches retraining unless the reported CKA and probe results support that statement.
"""
    (aggregate_dir / "professor_handoff.md").write_text(narrative, encoding="utf-8")

    manifest = {
        "dataset": cfg.dataset,
        "seed": seed,
        "experiment_dir": str(experiment_dir(cfg)),
        "single_seed_only": True,
        "methods": METHOD_ORDER,
        "primary_method": "scrubref_rep",
        "generated_at": utc_now(),
        "artifacts": sorted(str(path.relative_to(aggregate_dir)) for path in aggregate_dir.rglob("*") if path.is_file()),
    }
    atomic_write_json(aggregate_dir / "results_manifest.json", manifest)

    print("Single-seed publication package:", aggregate_dir)
    display(main_table)
    display(readiness)
    return {
        "aggregate_dir": aggregate_dir,
        "summary": aggregate_dir / "single_seed_method_summary.csv",
        "main_table": aggregate_dir / "paper_table_main.csv",
        "representation_table": aggregate_dir / "paper_table_representation.csv",
        "readiness": aggregate_dir / "reviewer_readiness_checks.csv",
        "handoff": aggregate_dir / "professor_handoff.md",
    }


## Stage 8 — professor handoff and downloadable result package

Run this cell after Stage 7. It creates a lightweight ZIP containing final tables, figures, result manifests, status files, and per-method CSV/JSON evidence. Large model and feature files remain in Google Drive and are not downloaded.

In [ ]:
from google.colab import files
import zipfile

PUBLICATION_PACKAGE = build_single_seed_publication_package(CFG, seed=42)

run_dir = seed_dir(CFG, 42)
aggregate_dir = PUBLICATION_PACKAGE["aggregate_dir"]
zip_path = Path(f"./results/{CFG.dataset}_seed42_fast_publication_package_v8.zip")

with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as archive:
    # Final tables, figures, manifest, and professor handoff.
    for path in aggregate_dir.rglob("*"):
        if path.is_file():
            archive.write(path, Path("aggregate_seed42") / path.relative_to(aggregate_dir))

    # Lightweight per-method evidence. Models and feature arrays are intentionally excluded.
    for folder_name in ("metrics", "status", "stages", "proxies"):
        folder = run_dir / folder_name
        if not folder.exists():
            continue
        for path in folder.rglob("*"):
            if path.is_file() and path.suffix.lower() in {".csv", ".json", ".txt", ".log"}:
                archive.write(path, Path(f"seed_42/{folder_name}") / path.relative_to(folder))

    for path in (run_dir / "config.json", run_dir / "representation_metrics.csv"):
        if path.exists():
            archive.write(path, Path("seed_42") / path.name)

print("Final publication ZIP:", zip_path)
print("This package excludes large .keras and .npz files.")
files.download(str(zip_path))

# Additional intervention ablations are optional and are not needed for the current handoff.
RUN_ADDITIONAL_MECHANISM_ABLATIONS = False
print("Additional mechanism ablations enabled:", RUN_ADDITIONAL_MECHANISM_ABLATIONS)


## Completion criteria

The dataset is complete when Stage 8 downloads the publication ZIP without reporting incomplete methods. The paper must describe these as fixed-seed results. Multi-seed validation can be added later if time permits.